<div align="center">

# Hafta 4: Transfer öğrenme

**Python ile Yapay Zekâ Eğitimi: Kardiyologlar için Uygulamalı Eğitim Programı**<br>
Türk Kardiyoloji Derneği · Prof. Dr. Utku Köse

[Hafta sayfası](README.md) · [Ders sayfası](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-04/ders.html) · [Etkileşimli lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-04/lab.html) · [PDF ders notu](Hafta04_Ders_Notlari.pdf) · [Colab'da aç](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-04/NB04_transfer_ogrenme.ipynb)

</div>

**Bu defter iki kısımdan oluşur.** Birincisinde kodu bir yapay zekâ (YZ) asistanı yazar; asistana ne yapacağını yazılı bir talimat, yani bir prompt (istem) anlatır. İkincisinde kodu Python ile kendiniz yazarsınız. İki kısım da aynı verilerle aynı sonuçlara varır; ders sayfasındaki her uygulama durağında bunlardan birini seçin.

| | Kimler için? | Ne yapılır? |
|---|---|---|
| **[Kısım 1: Prompt yolu](#part-1)** | Kod yazmayan katılımcılar | Kodu, hazır promptlarla bir YZ asistanı yazar; siz çalıştırır ve sonucu denetlersiniz. |
| **[Kısım 2: Python yolu](#part-2)** | Python öğrenmek isteyen katılımcılar | Kodu alıştırmalarla adım adım kendiniz yazar ve çalıştırırsınız. |

**Nerede çalışır?** Aynı defter, hiçbir kurulum gerektirmeyen Google Colab'da da kendi bilgisayarınızdaki Jupyter'de de (Anaconda; ders klasöründeki SETUP.tr.md dosyasına bakın) çalışır. Hangisini tercih ederseniz onu kullanın. Colab'da önce *Dosya → Drive'a kopya kaydet* (*File → Save a copy in Drive*) komutuyla kendi kopyanızı kaydedin; aksi hâlde sekme kapanınca yaptığınız değişiklikler kaybolur. Dört milyon ağırlıklı bir ağa ince ayar yapmak yaklaşık bir dakika, U-Net'i eğitmek ise yaklaşık üç dakika sürer; bunun için sıradan bir dizüstü bilgisayar yeterlidir ve ekran kartı gerekmez. Hangi kısmı seçerseniz seçin önce 0. bölümü çalıştırın. Bu bölüm ders dosyalarını ve paketleri hazırlar.

**Veri ve mahremiyet.** Buradaki bütün veri setleri ya sentetiktir ya da açık bir araştırma veri tabanından gelmektedir. Gerçek hasta verisi yalnızca kendi bilgisayarınızda, kurumunuzun kurallarına uygun olarak ve etik kurul onayıyla işlenir; bulut tabanlı bir deftere ya da bir YZ sohbetine asla konmaz.

## 0. Kurulum

İlk hücre ders klasörünü bulur. Defter tek başına açıldıysa, örneğin Google Colab'da, `vclinic` ders paketini ve veri dosyalarını GitHub'dan indirir. İkinci hücre paketleri yükler, mesajların dilini seçer ve haftanın isteğe bağlı paketlerini denetler. İkinci hücre eksik olanları indirir: 3. haftanın ACDC dosyalarını (yaklaşık 105 MB), Human Heart Project'in (CREATIS, Lyon) açık veri tabanından CAMUS'un 300 hastasının apikal ekokardiyogramlarını uzman sınırlarıyla birlikte (yaklaşık 227 MB) ve EfficientNet-B0'ın ImageNet ağırlıklarını (yaklaşık 21 MB). İlk çalıştırma birkaç dakika sürer; sonraki çalıştırmalarda zaten var olan dosyalar atlanır. ACDC ve CAMUS, ticari olmayan bilimsel araştırmalar için CC BY-NC-SA 4.0 lisansıyla ücretsiz olarak sunulmaktadır. CAMUS'un her kullanımında Leclerc ve arkadaşlarının IEEE Transactions on Medical Imaging dergisindeki 2019 tarihli makalesine atıf yapılmalıdır.

In [ ]:
#@title Ders dosyaları: Bu hücreyi çalıştırın (açmanıza gerek yok)
import pathlib
import sys
import urllib.request

REPO_RAW = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/"
DATA_FILES = [
]


def find_course_root():
    here = pathlib.Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "vclinic" / "__init__.py").exists():
            return folder
    return None


ROOT = find_course_root()
if ROOT is None or ROOT.name == "ai-cardiology-course":
    # İndirilmiş bir kopyada, örneğin Colab'da, paketin dosyaları yenilenir.
    ROOT = pathlib.Path.cwd() / "ai-cardiology-course" if ROOT is None else ROOT
    manifest = urllib.request.urlopen(REPO_RAW + "vclinic/MANIFEST.txt").read().decode("utf-8")
    package = ["vclinic/" + line.strip() for line in manifest.splitlines() if line.strip()]
    for rel in package + DATA_FILES:
        target = ROOT / rel
        if rel in package or not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(REPO_RAW + rel, target)
    print("Ders dosyaları şu klasöre indirildi:", ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
# Kurulum: paketler, dil ve çizim biçimi
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import copy
import math

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from vclinic import acdc, camus

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
except ImportError as error:
    raise ImportError("PyTorch ya da torchvision kurulu değil. SETUP.tr.md dosyasında anlatıldığı gibi kurun, ardından çekirdeği yeniden başlatın.") from error
try:
    import skimage
except ImportError as error:
    raise ImportError("scikit-image kurulu değil. pip install scikit-image komutuyla kurun, ardından çekirdeği yeniden başlatın.") from error
import vclinic
from vclinic import teach

if tuple(int(x) for x in vclinic.__version__.split(".")) < (1, 5, 0):
    raise ImportError("vclinic ders paketi bu defterden daha eskidir. Ders klasörünü güncelleyin (git pull komutuyla ya da ZIP dosyasını yeniden indirerek) ve çekirdeği yeniden başlatın.")
teach.set_language("tr")
plt.rcParams.update({"figure.dpi": 110, "font.family": "DejaVu Sans"})
RANDOM_STATE = 2026
torch.manual_seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))
print("PyTorch", torch.__version__)
GROUPS = ("NOR", "DCM", "HCM")
acdc.download(GROUPS)
camus.download()
EfficientNet_B0_Weights.IMAGENET1K_V1.get_state_dict(progress=False)
print("EfficientNet-B0'ın ImageNet ağırlıkları hazır.")
print("vclinic sürümü", vclinic.__version__)
print("Ders klasörü:", ROOT)
for name in ["ipywidgets"]:
    try:
        importlib.import_module(name)
        print(f"{name}: kurulu")
    except ImportError:
        print(f"{name}: kurulu değil (isteğe bağlı; defter yerleşik bir yedek yol kullanır)")

In [ ]:
#@title Alıştırmaların beklenen cevapları (bu hücreyi çalıştırın; açmayı alıştırmaları denedikten sonraya bırakın)
teach.register_answers({
    "Alıştırma 4.1": (46657, 0),
    "Alıştırma 4.2": (28.8, 0.05),
    "Alıştırma 4.3": (3843, 0),
    "Alıştırma A": (5288548, 0),
    "Alıştırma B": (1.0, 0.001),
    "Alıştırma C": (135, 0),
    "Alıştırma D": (0.687, 0.001),
    "Alıştırma E": (8.7, 0.05),
    "Alıştırma F": (0.789, 0.001),
    "Alıştırma G": (483170, 0),
    "Alıştırma H": (79.0, 0),
    "Alıştırma I": (7.0, 0.05),
    "Alıştırma J": (159.0, 0),
    "Adım 1.1": (0.867, 0.034),
    "Adım 1.2": (0.967, 0.1),
    "Adım 1.3": (0.62, 0.12),
    "Adım 2.1": (0.877, 0.001),
    "Adım 2.2": (0.924, 0.05),
    "Adım 2.3": (0.908, 0.03),
    "Adım 3.1": (42.1, 0.05),
    "Adım 3.2": (-4.24, 2.5),
    "Adım 3.3": (50.3, 0.05),
})
print("Beklenen cevaplar hazır.")

In [ ]:
#@title Alıştırma promptlarının tabloları (bu hücreyi çalıştırın; Python yolunda okunması gerekmez)
def _disc_rows(names):
    """Her hastanın dört uzman sınırının 20 diski; karelerin özgün boyutunda."""
    rows = []
    for name in names:
        for view in ("2CH", "4CH"):
            for phase in ("ED", "ES"):
                _, gt, spacing = camus.load(name, view, phase)
                diameters, thickness, _, _ = camus.discs(gt == 1, spacing)
                for k, d in enumerate(diameters, start=1):
                    rows.append({"hasta": name, "görüntü": view, "faz": phase, "disk": k,
                                 "çap": float(d), "uzun eksen": float(thickness * 20)})
    return pd.DataFrame(rows)


def alistirma_tablosu(anahtar):
    """Bir alıştırma promptunun "veri" tablosu; Python yolunun verilerinden kurulur."""
    g = globals()
    k = anahtar
    if k == "4.1":
        return pd.DataFrame(camus.load("patient0201", "4CH", "ED")[1])
    if k == "4.2":
        return pd.DataFrame(camus.load("patient0201", "4CH", "ES")[1])
    if k == "B":
        names = {"train": "eğitim", "val": "doğrulama", "test": "test"}
        f = g["features"]
        return pd.concat([pd.DataFrame({"hasta": g["patients"], "grup": [["NOR", "DCM", "HCM"][c] for c in g["y"]],
                                        "küme": [names[s] for s in g["sets"]]}),
                          pd.DataFrame(f, columns=[f"ö{i}" for i in range(1, f.shape[1] + 1)])], axis=1)
    if k == "D":
        return pd.DataFrame([{"grup başına hasta": n, "çekiliş": d, "doğruluk": a}
                             for n, accs in g["curve"].items() for d, a in enumerate(accs, start=1)])
    if k == "E":
        names = {"training": "eğitim", "validation": "doğrulama", "testing": "test"}
        return pd.DataFrame({"hasta": g["frame_patient"], "görüntü": g["frame_view"], "faz": g["frame_phase"],
                             "küme": [names[s] for s in g["frame_split"]], "sınır": list(g["masks"])})
    if k == "F":
        return pd.DataFrame({"diyastol sonu": [camus.load("patient0201", "4CH", "ED")[1]],
                             "sistol sonu": [camus.load("patient0201", "4CH", "ES")[1]]})
    if k in ("H", "J"):
        return _disc_rows(["patient0201"]).drop(columns="hasta")
    if k == "I":
        if "_discs_test_cache" not in g:
            g["_discs_test_cache"] = _disc_rows(camus.patients("testing"))
        return g["_discs_test_cache"]
    raise KeyError(k)
print("Alıştırma promptlarının tabloları hazır.")

<a id="part-1"></a>

# Kısım 1: Prompt yolu

**Kimler için?** Bu kısım kod yazmayan katılımcılar içindir. Burada hiç kod yazmazsınız. Kodu, hazır bir prompttan, yani yazılı bir talimattan yararlanarak bir yapay zekâ (YZ) asistanı yazar; siz kodu çalıştırır ve sonucu denetlersiniz.

**Neye ihtiyacınız var?** Python kodu yazabilen bir YZ asistanı yeterlidir. Google Colab'ın içinde Gemini bulunur; pencerenin üstündeki Gemini düğmesiyle açılır. Başka bir tarayıcı sekmesinde açılan ChatGPT, Claude, Copilot ya da Gemini de aynı işi görür.

**Bir adım nasıl ilerler?**

1. Adımın amacını okuyun.
2. Promptun altındaki veri hücresini çalıştırın. Hücre, promptun tarif ettiği "veri" tablosunu hazırlar ve tablonun ilk ve son satırlarını gösterir.
3. Promptu kopyalayın: Fareyle seçin ve Ctrl+C'ye (Mac'te Cmd+C) basın.
4. Promptu YZ asistanına yapıştırın ve gönderin.
5. Asistanın verdiği kodu kopyalayın, veri hücresinin altındaki hücreye `#` ile başlayan satırın yerine yapıştırın ve hücreyi ▶ düğmesiyle ya da Shift+Enter ile çalıştırın.
6. Hemen altındaki kontrol hücresini çalıştırın. Hücre *doğru ✓* ya da *henüz değil ✗* yazar.
7. *Henüz değil* yazarsa aşağıdaki onarım promptunu kodla birlikte asistana gönderin. Sonuç yine yanlışsa *Hazır kod* hücresini çalıştırın. Bu hücre her zaman doğru sonucu verir; böylece bir sonraki adıma geçebilirsiniz.

**"veri" ve "cevap".** 1. haftada olduğu gibi her adımın veri hücresi, promptun tarif ettiği "veri" tablosunu hazırlar. Asistan sonucu "cevap" adıyla saklar; kontrol hücresi de sonucu bu addan okur.

**Her durağın başında** o durağın hazırlık hücresini bir kez çalıştırın. Hücre durağın verilerini yükler; böylece bir durağa başka bir gün de başlanabilir.

**Onarım promptu** (çalışan ama kontrolden geçmeyen kod için):

> Aşağıdaki kod şu görev için yazıldı: [promptun Görev satırını yapıştırın]. Kod çalışıyor, ancak dersin kontrolü "henüz değil" yazıyor. Hatayı bul, iki cümleyle açıkla ve düzeltilmiş kodu ver. Sonuç yine "cevap" adıyla saklanmalı.  
>
> [kodu buraya yapıştırın]  

**Hata mesajı için prompt** (hata vererek duran kod için):

> Aşağıdaki kod bir Jupyter defterinde çalıştırıldığında altta verilen hata mesajını veriyor. Hatanın nedenini sade bir dille açıkla ve düzeltilmiş kodu ver.  
>
> [kodu buraya yapıştırın]  
>
> [hata mesajını buraya yapıştırın]  

Haftanın bütün promptları [prompt kütüphanesinde](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-04/PROMPTLAR.md) de yer alır; GitHub her promptun yanında bir kopyalama düğmesi gösterir.

**Hiç değişmeyen kural.** Promptlarda yalnızca dersin sentetik ya da açık verileri bulunur. Bir YZ asistanına gerçek hasta verisi asla yapıştırılmaz.

## Durak 1: Fotoğraflarla eğitilmiş bir ağla az hastadan öğrenmek

*Ders: [1. uygulama durağı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-04/ders.html#stop-1)* · *Python yolunda aynı sonuç: Kısım 2, 1-4. bölümler*

ImageNet'in bir milyondan fazla fotoğrafıyla eğitilmiş bir ağ kenarları, dokuları ve biçimleri zaten tanır. Bu durakta EfficientNet-B0, ACDC'nin kardiyomiyopati grupları üzerinde iki yoldan yeniden kullanılır: Ağın öznitelikleri basit bir modele verilir ve ağın tamamına ince ayar yapılır. Ardından her gruptan yalnızca beş hasta tutulur ve sonucun, rastgele seçilen hastalara ne kadar bağlı olduğu görülür.

**Durağın sonunda:** Özniteliklerle kurulan lojistik regresyonun 30 test hastasındaki doğruluğunu (0,867), ince ayar yapılmış ağın doğruluğunu (yaklaşık 0,967) ve her gruptan beş hastayla elde edilen ortalama doğruluğu (0,62) bulacaksınız. İnce ayar yaklaşık bir dakika sürer.

In [ ]:
#@title 1. durağın hazırlığı: Bu hücreyi bir kez çalıştırın (durağın verilerini yükler)
GROUPS = ("NOR", "DCM", "HCM")
P = acdc.pairs(GROUPS)
X, y, patients, sets, classes = P["X"], P["y"], P["patient"], P["set"], P["classes"]

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_SD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def as_photos(x, size=128):
    """ED, ES ve ortalamaları üç renk kanalı olarak; ImageNet'teki gibi boyutlandırılmış ve normalleştirilmiş."""
    x = torch.as_tensor(x, dtype=torch.float32)
    x3 = torch.stack([x[:, 0], x[:, 1], (x[:, 0] + x[:, 1]) / 2], dim=1)
    x3 = F.interpolate(x3, size=(size, size), mode="bilinear", align_corners=False)
    return (x3 - IMAGENET_MEAN) / IMAGENET_SD

def by_patient(prob, mask):
    """Her hastanın kesitleri üzerinden ortalama olasılıklar ve doğru kararların oranı."""
    names = sorted(set(patients[mask]))
    mean_prob = np.array([prob[patients[mask] == n].mean(axis=0) for n in names])
    truth = np.array([y[mask][patients[mask] == n][0] for n in names])
    return float((mean_prob.argmax(axis=1) == truth).mean())


def _course_features():
    """EfficientNet-B0'ın 270 kesit için verdiği 1.280 öznitelik (bir kez hesaplanır)."""
    if "_features_cache" not in globals():
        net = efficientnet_b0(weights=effnet_weights).eval()
        with torch.no_grad():
            globals()["_features_cache"] = torch.cat([net.avgpool(net.features(photos[i:i + 32])).flatten(1)
                                                      for i in range(0, len(photos), 32)]).numpy()
    return _features_cache

photos = as_photos(X)
effnet_weights = EfficientNet_B0_Weights.IMAGENET1K_V1
_SETS = {"train": "eğitim", "val": "doğrulama", "test": "test"}
print(tuple(photos.shape), classes, {s: int((sets == s).sum()) for s in ("train", "val", "test")})
print("1. durak hazır.")

### Adım 1.1: Fotoğraflarla eğitilmiş bir ağın öznitelikleri

**Amaç.** Bu adımda her kesit EfficientNet-B0'ın 1.280 özniteliğiyle tarif edilir ve üç grup bu özniteliklerden basit bir modelle öğrenilir.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: ACDC'nin kardiyomiyopati gruplarındaki sine MR kesitlerinde, yalnızca fotoğraflarla eğitilmiş bir ağın özniteliklerinin normal kalpleri, dilate kardiyomiyopatiyi ve hipertrofik kardiyomiyopatiyi ne kadar iyi ayırdığını öğrenmek istiyorum.  
> Veri: Defterde "veri" adında bir tablo var; ACDC'deki 90 hastanın her birinin ortadaki üç kısa eksen kesitinden oluşan 270 sine MR kesitini tutar. Her satır bir kesittir: "hasta" sütunu hastayı, "grup" hastanın grubunu ("NOR", "DCM" ya da "HCM"), "küme" ise hastanın kümesini verir ("eğitim", her gruptan 15 olmak üzere 45 hasta; "doğrulama", 15 hasta; "test", 30 hasta). "görüntü" sütununun her hücresi, kesiti fotoğraflarla eğitilmiş bir ağa uygun biçimde tutar: 128 × 128 piksellik üç renk kanalı (diyastol sonu karesi, sistol sonu karesi ve ikisinin ortalaması), ImageNet'in ortalamaları ve standart sapmalarıyla normalleştirilmiştir.  
> Görev: EfficientNet-B0'ı ImageNet ağırlıklarıyla ve değerlendirme modunda kullan. Bütün kesitlerin görüntülerini gradyan kaydetmeden, 32'şerlik gruplar hâlinde ağdan geçir ve ağın son katmanından önceki ortalama havuzlamanın çıktısını, yani kesit başına 1.280 sayıdan oluşan öznitelikleri al. Öznitelikleri eğitim kesitleriyle standartlaştır ve eğitim kesitlerinde bir lojistik regresyon eğit. Ardından test kesitleri için üç grubun olasılıklarını hesapla, bu olasılıkların her test hastasının üç kesiti üzerinden ortalamasını al ve ortalaması en büyük grubu hastanın kararı olarak seç.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Ağ için yalnızca bu bilgisayarda hazır bulunan EfficientNet-B0 ImageNet ağırlıklarını kullan; yeni bir indirme gerekmez. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: 30 test hastasından kararı kendi grubuyla aynı olanların oranını üç ondalık basamakla yazdır ve "cevap" adıyla sakla.  
> Test: Öznitelikler 270 satır ve 1.280 sütundan oluşan bir tablo olmalı ve 30 test hastası bulunmalı.  

In [ ]:
#@title Adım 1.1 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_SETS = {"train": "eğitim", "val": "doğrulama", "test": "test"}
veri = pd.DataFrame({"hasta": patients, "grup": [classes[k] for k in y],
                     "küme": [_SETS[s] for s in sets], "görüntü": list(photos.numpy())})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 1.1 kontrolü
teach.check_step("Adım 1.1", "cevap", globals())

In [ ]:
#@title Adım 1.1 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_SETS = {"train": "eğitim", "val": "doğrulama", "test": "test"}
veri = pd.DataFrame({"hasta": patients, "grup": [classes[k] for k in y],
                     "küme": [_SETS[s] for s in sets], "görüntü": list(photos.numpy())})


def hasta_basina_dogruluk(olasilik, adlar, gercek):
    tablo = pd.DataFrame(olasilik)
    tablo["hasta"] = adlar
    ortalamalar = tablo.groupby("hasta").mean()
    gercek_grup = pd.Series(gercek, index=adlar).groupby(level=0).first()
    return float((ortalamalar.to_numpy().argmax(axis=1) == gercek_grup.loc[ortalamalar.index].to_numpy()).mean())


kesit_goruntuleri = torch.tensor(np.stack(veri["görüntü"]))
ag = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1).eval()
with torch.no_grad():
    kesit_oznitelikleri = torch.cat([ag.avgpool(ag.features(kesit_goruntuleri[i:i + 32])).flatten(1)
                                for i in range(0, len(kesit_goruntuleri), 32)]).numpy()
kesit_kumeleri = veri["küme"].to_numpy()
gruplar = veri["grup"].to_numpy()
egitimde, testte = kesit_kumeleri == "eğitim", kesit_kumeleri == "test"
lojistik = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=5000)).fit(kesit_oznitelikleri[egitimde], gruplar[egitimde])
test_olasiligi = lojistik.predict_proba(kesit_oznitelikleri[testte])
test_kodlari = pd.Categorical(gruplar[testte], categories=lojistik.classes_).codes
test_adlari = veri["hasta"].to_numpy()[testte]
cevap = round(hasta_basina_dogruluk(test_olasiligi, test_adlari, test_kodlari), 3)
print(kesit_oznitelikleri.shape, "test hastası:", len(set(test_adlari)), "doğruluk:", cevap)
teach.check_step("Adım 1.1", "cevap", globals())

**Sonuç ne gösteriyor?** Yalnızca fotoğraflardan öğrenilmiş öznitelikler, 30 test hastasının yüzde 87 kadarını kendi grubuna yerleştirir. Ağ hiç kalp görmemiştir; ilk katmanları kenarlara ve karşıtlıklara tepki verir. Lojistik regresyon ise 1.280 özniteliğin hangi birleşimlerinin dilate, kalın duvarlı ve normal bir ventrikülü birbirinden ayırdığını öğrenir.

### Adım 1.2: Ağın tamamına ince ayar

**Amaç.** Bu adımda EfficientNet-B0'ın bütün katmanları, ImageNet ağırlıklarından başlayarak kesitlere uyarlanır. Eğitim yaklaşık bir dakika sürer.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Fotoğraflarla eğitilmiş bir ağın bütün katmanlarını ACDC'nin sine MR kesitlerine uyarlamanın, üç grubu yalnızca ağın özniteliklerinden daha iyi ayırıp ayırmadığını öğrenmek istiyorum.  
> Veri: Defterde "veri" adında bir tablo var; ACDC'deki 90 hastanın her birinin ortadaki üç kısa eksen kesitinden oluşan 270 sine MR kesitini tutar. Her satır bir kesittir: "hasta" sütunu hastayı, "grup" hastanın grubunu ("NOR", "DCM" ya da "HCM"), "küme" ise hastanın kümesini verir ("eğitim", her gruptan 15 olmak üzere 45 hasta; "doğrulama", 15 hasta; "test", 30 hasta). "görüntü" sütununun her hücresi, kesiti fotoğraflarla eğitilmiş bir ağa uygun biçimde tutar: 128 × 128 piksellik üç renk kanalı (diyastol sonu karesi, sistol sonu karesi ve ikisinin ortalaması), ImageNet'in ortalamaları ve standart sapmalarıyla normalleştirilmiştir.  
> Görev: EfficientNet-B0'ı ImageNet ağırlıklarıyla kur ve son katmanını, 0,3 oranında dropout'un ardından gelen ve her grup için bir çıktı veren üç çıktılı yeni bir doğrusal katmanla değiştir. Ağın tamamını eğitim kesitlerinde 15 epoch boyunca, 16 kesitlik karıştırılmış gruplarla, 0,001 öğrenme hızıyla ve çapraz entropi kaybıyla eğit. Veri çoğaltma için her kesit grubunu rastgele bir çeyrek tur döndür. Her epoch'un sonunda, onlardan öğrenmeden doğrulama kesitlerindeki kaybı ölç ve bu kaybın en düşük olduğu epoch'un ağırlıklarını sakla. Rastgele seçimler her çalıştırmada aynı olsun; bunun için başlangıç değeri olarak 2026 sayısını kullan. Ardından her test hastası için bir kliniğin yapacağı gibi karar ver: Üç grubun olasılıklarının hastanın üç kesiti üzerinden ortalamasını al ve ortalaması en büyük grubu seç.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Ağ için yalnızca bu bilgisayarda hazır bulunan EfficientNet-B0 ImageNet ağırlıklarını kullan; yeni bir indirme gerekmez. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Doğrulama kaybının en düşük olduğu epoch'u ve 30 test hastasından kararı kendi grubuyla aynı olanların oranını üç ondalık basamakla yazdır; oranı "cevap" adıyla sakla.  
> Test: Saklanan ağırlıklar yüklendikten sonra doğrulama kaybı yine en düşük değerini vermeli.  

In [ ]:
#@title Adım 1.2 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_SETS = {"train": "eğitim", "val": "doğrulama", "test": "test"}
veri = pd.DataFrame({"hasta": patients, "grup": [classes[k] for k in y],
                     "küme": [_SETS[s] for s in sets], "görüntü": list(photos.numpy())})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 1.2 kontrolü
teach.check_step("Adım 1.2", "cevap", globals())

In [ ]:
#@title Adım 1.2 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_SETS = {"train": "eğitim", "val": "doğrulama", "test": "test"}
veri = pd.DataFrame({"hasta": patients, "grup": [classes[k] for k in y],
                     "küme": [_SETS[s] for s in sets], "görüntü": list(photos.numpy())})


def hasta_basina_dogruluk(olasilik, adlar, gercek):
    tablo = pd.DataFrame(olasilik)
    tablo["hasta"] = adlar
    ortalamalar = tablo.groupby("hasta").mean()
    gercek_grup = pd.Series(gercek, index=adlar).groupby(level=0).first()
    return float((ortalamalar.to_numpy().argmax(axis=1) == gercek_grup.loc[ortalamalar.index].to_numpy()).mean())


kesit_goruntuleri = torch.tensor(np.stack(veri["görüntü"]))
kodlar = torch.tensor(veri["grup"].map({"NOR": 0, "DCM": 1, "HCM": 2}).to_numpy())
kesit_kumeleri = veri["küme"].to_numpy()
egitimde, dogrulamada, testte = kesit_kumeleri == "eğitim", kesit_kumeleri == "doğrulama", kesit_kumeleri == "test"
torch.manual_seed(2026)
ayarli_ag = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
ayarli_ag.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(1280, 3))
optimizer = torch.optim.Adam(ayarli_ag.parameters(), lr=1e-3, weight_decay=1e-4)
sira_ureteci = torch.Generator().manual_seed(2026)
x_tr, y_tr = kesit_goruntuleri[egitimde], kodlar[egitimde]
best, best_state = float("inf"), None
for epoch in range(15):
    ayarli_ag.train()
    order = torch.randperm(len(y_tr), generator=sira_ureteci)
    for start in range(0, len(order), 16):
        j = order[start:start + 16]
        turns = int(torch.randint(0, 4, (1,), generator=sira_ureteci))
        loss = F.cross_entropy(ayarli_ag(torch.rot90(x_tr[j], turns, dims=(2, 3))), y_tr[j])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    ayarli_ag.eval()
    with torch.no_grad():
        val_loss = F.cross_entropy(ayarli_ag(kesit_goruntuleri[dogrulamada]), kodlar[dogrulamada]).item()
    if val_loss < best:
        best, best_state, best_epoch = val_loss, copy.deepcopy(ayarli_ag.state_dict()), epoch + 1
ayarli_ag.load_state_dict(best_state)
ayarli_ag.eval()
with torch.no_grad():
    kesit_olasiligi = F.softmax(ayarli_ag(kesit_goruntuleri[testte]), dim=1).numpy()
cevap = round(hasta_basina_dogruluk(kesit_olasiligi, veri["hasta"].to_numpy()[testte], kodlar.numpy()[testte]), 3)
print("En iyi epoch:", best_epoch, "en düşük doğrulama kaybı:", round(best, 4), "doğruluk:", cevap)
teach.check_step("Adım 1.2", "cevap", globals())

**Sonuç ne gösteriyor?** Dersin çalıştırmasında ince ayar yaklaşık 0,967 doğruluğa ulaşmıştır; bu, haftanın bu 30 hastadaki en iyi sonucudur. Eğitim bilgisayardan bilgisayara biraz değiştiği için kontrol, bu değerin 0,1 yakınındaki sonuçları kabul etmektedir. Ağın her ağırlığı fotoğraflardan başlamış ve 45 hastanın kesitlerine göre ayarlanmıştır.

### Adım 1.3: Her gruptan beş hasta

**Amaç.** Bu adımda az sayıda hastadan elde edilen bir sonucun, seçilen hastalara ne kadar bağlı olduğu görülür.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Her gruptan yalnızca beş hastayla elde edilen bir sonucun, rastgele seçilen hastalara ne kadar bağlı olduğunu görmek istiyorum.  
> Veri: Defterde "veri" adında bir tablo var; ACDC'deki 90 hastanın her birinin ortadaki üç kısa eksen kesitinden oluşan 270 sine MR kesitini tutar. Her satır bir kesittir: "hasta" sütunu hastayı, "grup" hastanın grubunu ("NOR", "DCM" ya da "HCM"), "küme" ise hastanın kümesini verir ("eğitim", her gruptan 15 olmak üzere 45 hasta; "doğrulama", 15 hasta; "test", 30 hasta). "ö1"den "ö1280"e kadar olan öteki 1.280 sütun, kesitin özniteliklerini, yani ImageNet fotoğraflarıyla eğitilmiş EfficientNet-B0'ın son katmanından önceki ortalama havuzlamanın çıktısını tutar.  
> Görev: Eğitim kümesinin her grubundan rastgele beşer hasta seç ve bu seçimi beş kez tekrarla. Rastgele seçimler her çalıştırmada aynı olsun; bunun için başlangıç değeri olarak 2026 sayısını kullan. Her seçimde öznitelikleri yalnızca seçilen 15 hastanın kesitleriyle standartlaştır ve bu kesitlerde bir lojistik regresyon eğit. Ardından 30 test hastasındaki doğruluğu, her hasta için tek bir kararla hesapla: Kararı, hastanın üç kesiti üzerinden ortalama olasılığı en büyük olan grup belirler.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Beş doğruluğu yazdır; ortalamalarını üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Her seçimde, her gruptan beşer olmak üzere 15 farklı hasta bulunmalı.  

In [ ]:
#@title Adım 1.3 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_SETS = {"train": "eğitim", "val": "doğrulama", "test": "test"}
_F = _course_features()
veri = pd.concat([pd.DataFrame({"hasta": patients, "grup": [classes[k] for k in y], "küme": [_SETS[s] for s in sets]}),
                  pd.DataFrame(_F, columns=[f"ö{k}" for k in range(1, _F.shape[1] + 1)])], axis=1)
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 1.3 kontrolü
teach.check_step("Adım 1.3", "cevap", globals())

In [ ]:
#@title Adım 1.3 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_SETS = {"train": "eğitim", "val": "doğrulama", "test": "test"}
_F = _course_features()
veri = pd.concat([pd.DataFrame({"hasta": patients, "grup": [classes[k] for k in y], "küme": [_SETS[s] for s in sets]}),
                  pd.DataFrame(_F, columns=[f"ö{k}" for k in range(1, _F.shape[1] + 1)])], axis=1)


def hasta_basina_dogruluk(olasilik, adlar, gercek):
    tablo = pd.DataFrame(olasilik)
    tablo["hasta"] = adlar
    ortalamalar = tablo.groupby("hasta").mean()
    gercek_grup = pd.Series(gercek, index=adlar).groupby(level=0).first()
    return float((ortalamalar.to_numpy().argmax(axis=1) == gercek_grup.loc[ortalamalar.index].to_numpy()).mean())


oz = veri[[c for c in veri.columns if c.startswith("ö")]].to_numpy()
gruplar = veri["grup"].to_numpy()
adlar = veri["hasta"].to_numpy()
kesit_kumeleri = veri["küme"].to_numpy()
egitimde, testte = kesit_kumeleri == "eğitim", kesit_kumeleri == "test"
rng = np.random.default_rng(2026)
egitim_hastalari = sorted(set(adlar[egitimde]))
grubu = {n: gruplar[adlar == n][0] for n in egitim_hastalari}
dogruluklar = []
for draw in range(5):
    secilen = []
    for g in ("NOR", "DCM", "HCM"):
        havuz = [n for n in egitim_hastalari if grubu[n] == g]
        secilen += sorted(rng.choice(havuz, size=5, replace=False).tolist())
    rows = np.isin(adlar, secilen)
    kucuk = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=5000)).fit(oz[rows], gruplar[rows])
    kodlar = pd.Categorical(gruplar[testte], categories=kucuk.classes_).codes
    dogruluklar.append(hasta_basina_dogruluk(kucuk.predict_proba(oz[testte]), adlar[testte], kodlar))
    print("seçim", draw + 1, len(set(secilen)), "hasta, doğruluk", round(dogruluklar[-1], 3))
cevap = round(float(np.mean(dogruluklar)), 3)
print("Ortalama doğruluk:", cevap)
teach.check_step("Adım 1.3", "cevap", globals())

**Sonuç ne gösteriyor?** Her gruptan beş hastayla yapılan beş seçim ortalama 0,62 doğruluk verir ve beş doğruluk birbirinden oldukça uzaktır. Hasta sayısı az olduğunda sonuç, rastgele seçilen hastalara bağlıdır. Bu yüzden yöntemlerin adil bir karşılaştırmasında her yöntem aynı seçimleri kullanır ve sonuçların dağılımı da bildirilir; ders sayfası da böyle yapmaktadır.

**1. durak tamamlandı.** [Ders sayfasına](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-04/ders.html#stop-1) dönün ve duraktan sonrasını okumaya devam edin.

## Durak 2: CAMUS ekokardiyogramlarında bir U-Net

*Ders: [2. uygulama durağı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-04/ders.html#stop-2)* · *Python yolunda aynı sonuç: Kısım 2, 5-7. bölümler*

Bir segmentasyon ağı her piksele bir etiket verir. Bu durakta önce Dice katsayısı, birkaç piksel kaydırılmış bir sınır üzerinde elle hesaplanır. Ardından küçük bir U-Net, sol ventrikül kavitesini CAMUS'un 800 eğitim karesinden öğrenir ve 50 test hastasının 200 karesinde görüntüye ve faza göre değerlendirilir.

**Durağın sonunda:** Kaydırılmış bir sınırın Dice katsayısını (0,877), U-Net'in test karelerindeki ortalama Dice katsayısını (yaklaşık 0,924) ve en zayıf görüntü ve faz çiftini (yaklaşık 0,908) bulacaksınız. Eğitim yaklaşık üç dakika sürer.

In [ ]:
#@title 2. durağın hazırlığı: Bu hücreyi bir kez çalıştırın (durağın verilerini yükler)
if "frames_" not in globals():
    frames_ = camus.frames(camus.patients(), size=96)
images, masks = frames_["images"], frames_["masks"]
frame_patient, frame_view, frame_phase, frame_split = frames_["patient"], frames_["view"], frames_["phase"], frames_["split"]
frame_shape = frames_["shape"]

def unet_block(c_in, c_out):
    return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
                         nn.Conv2d(c_out, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU())


class UNet(nn.Module):
    """Üç düzey aşağı, bir dar boğaz ve atlama bağlantılarıyla üç düzey yukarı."""

    def __init__(self, c=8, n_classes=2):
        super().__init__()
        self.down1, self.down2, self.down3 = unet_block(1, c), unet_block(c, 2 * c), unet_block(2 * c, 4 * c)
        self.bottom = unet_block(4 * c, 8 * c)
        self.up3, self.dec3 = nn.ConvTranspose2d(8 * c, 4 * c, 2, stride=2), unet_block(8 * c, 4 * c)
        self.up2, self.dec2 = nn.ConvTranspose2d(4 * c, 2 * c, 2, stride=2), unet_block(4 * c, 2 * c)
        self.up1, self.dec1 = nn.ConvTranspose2d(2 * c, c, 2, stride=2), unet_block(2 * c, c)
        self.out = nn.Conv2d(c, n_classes, 1)

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(F.max_pool2d(d1, 2))
        d3 = self.down3(F.max_pool2d(d2, 2))
        b = self.bottom(F.max_pool2d(d3, 2))
        u3 = self.dec3(torch.cat([self.up3(b), d3], dim=1))
        u2 = self.dec2(torch.cat([self.up2(u3), d2], dim=1))
        u1 = self.dec1(torch.cat([self.up1(u2), d1], dim=1))
        return self.out(u1)


def soft_dice_loss(logits, target):
    p = F.softmax(logits, dim=1)[:, 1]
    t = target.float()
    inter = (p * t).sum(dim=(1, 2))
    return 1 - ((2 * inter + 1) / (p.sum(dim=(1, 2)) + t.sum(dim=(1, 2)) + 1)).mean()


def dice(pred, true):
    """Her karenin Dice katsayısı: aynı boyutta iki mantıksal dizi."""
    inter = (pred & true).sum(axis=(-2, -1))
    return 2 * inter / np.maximum(pred.sum(axis=(-2, -1)) + true.sum(axis=(-2, -1)), 1)


def _course_unet():
    """Dersin U-Net'i; bir kez eğitilir (yaklaşık üç dakika) ve 200 test karesinin her pikselinde kavite olasılığını verir."""
    if "_unet_cache" in globals():
        return _unet_cache
    cavity = masks == 1
    tr, va, te = frame_split == "training", frame_split == "validation", frame_split == "testing"
    X_img = torch.tensor(images)[:, None]
    Y_cav = torch.tensor(cavity.astype(np.int64))
    torch.manual_seed(RANDOM_STATE)
    unet = UNet()
    epochs, batch = 15, 16
    optimizer = torch.optim.Adam(unet.parameters(), lr=3e-3)
    steps = epochs * ((int(tr.sum()) + batch - 1) // batch)
    schedule = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=3e-3, total_steps=steps)
    order_gen = torch.Generator().manual_seed(RANDOM_STATE)
    X_tr, Y_tr = X_img[tr], Y_cav[tr]
    best, best_state = -1.0, None
    for epoch in range(epochs):
        unet.train()
        order = torch.randperm(len(X_tr), generator=order_gen)
        for start in range(0, len(X_tr), batch):
            j = order[start:start + batch]
            out = unet(X_tr[j])
            loss = F.cross_entropy(out, Y_tr[j]) + soft_dice_loss(out, Y_tr[j])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            schedule.step()
        unet.eval()
        with torch.no_grad():
            val_dice = float(dice(unet(X_img[va]).argmax(dim=1).numpy() == 1, cavity[va]).mean())
        if val_dice > best:
            best, best_state, best_epoch = val_dice, copy.deepcopy(unet.state_dict()), epoch + 1
    unet.load_state_dict(best_state)
    unet.eval()
    with torch.no_grad():
        test_prob = F.softmax(unet(X_img[te]), dim=1)[:, 1].numpy()
    globals()["_unet_cache"] = test_prob
    return test_prob


def _course_agreement():
    """50 test hastasının uzman sınırlarından ve dersin U-Net'inin sınırlarından ölçülen hacimleri ve EF'si (bir kez hesaplanır)."""
    if "_agreement_cache" in globals():
        return _agreement_cache
    prob = _course_unet()
    outlines = {}
    for k, i in enumerate(np.where(frame_split == "testing")[0]):
        outlines.setdefault(frame_patient[i], {})[(frame_view[i], frame_phase[i])] = camus.to_original(prob[k], frame_shape[i])
    rows = []
    for name in camus.patients("testing"):
        ref, net = camus.measure(name), camus.measure(name, outlines[name])
        rows.append({"patient": name, "edv_expert": ref["edv_ml"], "edv_net": net["edv_ml"], "esv_expert": ref["esv_ml"],
                     "esv_net": net["esv_ml"], "ef_expert": ref["ef"], "ef_net": net["ef"]})
    globals()["_agreement_cache"] = pd.DataFrame(rows)
    return _agreement_cache
_FSETS = {"training": "eğitim", "validation": "doğrulama", "testing": "test"}

print(images.shape, {s: int((frame_split == s).sum()) for s in ("training", "validation", "testing")})
print("2. durak hazır.")

### Adım 2.1: Elle Dice katsayısı

**Amaç.** Bu adımda bir sınırın küçük bir kaymasının Dice katsayısını ne kadar düşürdüğü görülür.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Bir sınırın küçük bir kaymasının, iki sınır arasındaki uyumun olağan ölçütü olan Dice katsayısını ne kadar düşürdüğünü görmek istiyorum.  
> Veri: Defterde "veri" adında 96 satır ve 96 sütunluk bir tablo var: CAMUS'taki 201 numaralı hastanın apikal dört boşluk görüntüsünün diyastol sonu karesinde uzmanın çizdiği sol ventrikül kavitesidir; kare 96 × 96 piksele küçültülmüştür. Her hücre bir pikseldir: Kavitenin içindeki pikseller 1, dışındakiler 0'dır.  
> Görev: Kaviteyi 3 piksel sağa kaydır, yani her pikseli 3 sütun sağa taşı. Özgün ve kaydırılmış kavitenin piksellerini ve ikisinin ortak piksellerini say. Dice katsayısını hesapla: Ortak piksellerin iki katı, iki sayının toplamına bölünür.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Üç sayıyı yazdır; Dice katsayısını üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Kavitenin kaydırılmamış kendisiyle Dice katsayısı 1 olmalı.  

In [ ]:
#@title Adım 2.1 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_i = int(np.where((frame_patient == "patient0201") & (frame_view == "4CH") & (frame_phase == "ED"))[0][0])
veri = pd.DataFrame((masks[_i] == 1).astype(int))
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 2.1 kontrolü
teach.check_step("Adım 2.1", "cevap", globals())

In [ ]:
#@title Adım 2.1 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_i = int(np.where((frame_patient == "patient0201") & (frame_view == "4CH") & (frame_phase == "ED"))[0][0])
veri = pd.DataFrame((masks[_i] == 1).astype(int))
a = veri.to_numpy() == 1
b = np.roll(a, 3, axis=1)
ortak = int((a & b).sum())
cevap = round(2 * ortak / (int(a.sum()) + int(b.sum())), 3)
print("piksel:", int(a.sum()), int(b.sum()), "ortak:", ortak, "Dice:", cevap,
      "kaydırmadan:", 2 * int(a.sum()) / (2 * int(a.sum())))
teach.check_step("Adım 2.1", "cevap", globals())

**Sonuç ne gösteriyor?** 96 piksellik bir karede yaklaşık 3 mm'ye karşılık gelen 3 piksellik bir kayma, Dice katsayısını 0,877 değerine düşürür. Küçük kaviteler daha çok kaybeder: Aynı kayma, sistol sonundaki bir kavitenin piksellerinin daha büyük bir kısmını etkiler. ES karelerinde Dice katsayısının daha düşük olmasının nedenlerinden biri budur.

### Adım 2.2: U-Net'i eğitmek

**Amaç.** Bu adımda sol ventrikül kavitesini çizen küçük bir U-Net eğitilir ve ağ, hiç görmediği hastaların karelerinde ölçülür. Eğitim yaklaşık üç dakika sürer.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: CAMUS'un apikal ekokardiyogramlarında sol ventrikül kavitesinin sınırını çizen küçük bir U-Net eğitmek ve bu ağı hiç görmediği hastalarda ölçmek istiyorum.  
> Veri: Defterde "veri" adında bir tablo var; CAMUS'un 1.200 karesini, yani 300 hastanın apikal iki boşluk ve dört boşluk görüntülerinin diyastol sonu ve sistol sonu karelerini tutar. Her satır bir karedir: "hasta" sütunu hastayı, "görüntü" görüntüyü ("2CH" ya da "4CH"), "faz" fazı ("ED" ya da "ES"), "küme" ise hastanın kümesini verir ("eğitim", 200 hasta; "doğrulama", 50 hasta; "test", 50 hasta). "kare" sütununun her hücresi kareyi değerleri 0 ile 1 arasında olan 96 × 96 piksellik bir görüntü olarak, "kavite" sütununun her hücresi ise uzmanın çizdiği sol ventrikül kavitesini aynı boyutta tutar: Kavitenin içindeki pikseller 1, dışındakiler 0'dır.  
> Görev: Her piksele, biri kavite, biri kavite dışı için olmak üzere iki skor veren küçük bir U-Net kur. Ağda atlama bağlantılarıyla üç düzey aşağı, bir dar boğaz ve üç düzey yukarı bulunur. Her düzeyde görüntünün boyutunu koruyan 3 × 3 çekirdekli iki evrişim vardır ve her evrişimin ardından batch normalizasyonu ile ReLU gelir. Aşağı inerken 2 × 2 maksimum havuzlama görüntüyü yarıya indirir; yukarı çıkarken 2 × 2'lik ters evrişim görüntüyü iki kat büyütür ve sonuç, aşağı inerken aynı düzeyde elde edilen çıktıyla birleştirilir. İlk düzeyde 8 kanal vardır ve kanal sayısı aşağı inilen her düzeyde iki katına çıkarak dar boğazda 64'e ulaşır. Son bir 1 × 1 evrişim iki skoru verir. Ağı eğitim karelerinde 15 epoch boyunca, 16 karelik karıştırılmış gruplarla eğit. Kayıp, çapraz entropi ile yumuşak Dice kaybının toplamıdır; yumuşak Dice kaybı, kavite olasılıklarından hesaplanan Dice katsayısının 1'den çıkarılmasıyla bulunur. En yüksek değeri 0,003 olan tek döngülü bir öğrenme hızı kullan. Her epoch'un sonunda doğrulama karelerinin ortalama Dice katsayısını hesapla ve bu değerin en yüksek olduğu epoch'un ağırlıklarını sakla. Rastgele seçimler her çalıştırmada aynı olsun; bunun için başlangıç değeri olarak 2026 sayısını kullan. Ardından test karelerinin her pikselinde kavite olasılığını hesapla, olasılığı 0,5'te eşikle ve her test karesinin Dice katsayısını uzmanın çizdiği kaviteye göre hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Eğitilebilir parametre sayısını yazdır; 200 test karesinin ortalama Dice katsayısını üç ondalık basamakla "cevap" adıyla sakla.  
> Test: 800 eğitim karesi ve 200 test karesi bulunmalı ve ağın 121.394 eğitilebilir parametresi olmalı.  

In [ ]:
#@title Adım 2.2 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_FSETS = {"training": "eğitim", "validation": "doğrulama", "testing": "test"}
veri = pd.DataFrame({"hasta": frame_patient, "görüntü": frame_view, "faz": frame_phase,
                     "küme": [_FSETS[s] for s in frame_split], "kare": list(images),
                     "kavite": list((masks == 1).astype(np.uint8))})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 2.2 kontrolü
teach.check_step("Adım 2.2", "cevap", globals())

In [ ]:
#@title Adım 2.2 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_FSETS = {"training": "eğitim", "validation": "doğrulama", "testing": "test"}
veri = pd.DataFrame({"hasta": frame_patient, "görüntü": frame_view, "faz": frame_phase,
                     "küme": [_FSETS[s] for s in frame_split], "kare": list(images),
                     "kavite": list((masks == 1).astype(np.uint8))})


def unet_block(c_in, c_out):
    return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
                         nn.Conv2d(c_out, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU())


class UNet(nn.Module):
    """Üç düzey aşağı, bir dar boğaz ve atlama bağlantılarıyla üç düzey yukarı."""

    def __init__(self, c=8, n_classes=2):
        super().__init__()
        self.down1, self.down2, self.down3 = unet_block(1, c), unet_block(c, 2 * c), unet_block(2 * c, 4 * c)
        self.bottom = unet_block(4 * c, 8 * c)
        self.up3, self.dec3 = nn.ConvTranspose2d(8 * c, 4 * c, 2, stride=2), unet_block(8 * c, 4 * c)
        self.up2, self.dec2 = nn.ConvTranspose2d(4 * c, 2 * c, 2, stride=2), unet_block(4 * c, 2 * c)
        self.up1, self.dec1 = nn.ConvTranspose2d(2 * c, c, 2, stride=2), unet_block(2 * c, c)
        self.out = nn.Conv2d(c, n_classes, 1)

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(F.max_pool2d(d1, 2))
        d3 = self.down3(F.max_pool2d(d2, 2))
        b = self.bottom(F.max_pool2d(d3, 2))
        u3 = self.dec3(torch.cat([self.up3(b), d3], dim=1))
        u2 = self.dec2(torch.cat([self.up2(u3), d2], dim=1))
        u1 = self.dec1(torch.cat([self.up1(u2), d1], dim=1))
        return self.out(u1)


def soft_dice_loss(logits, target):
    p = F.softmax(logits, dim=1)[:, 1]
    t = target.float()
    inter = (p * t).sum(dim=(1, 2))
    return 1 - ((2 * inter + 1) / (p.sum(dim=(1, 2)) + t.sum(dim=(1, 2)) + 1)).mean()


def dice(pred, true):
    """Her karenin Dice katsayısı: aynı boyutta iki mantıksal dizi."""
    inter = (pred & true).sum(axis=(-2, -1))
    return 2 * inter / np.maximum(pred.sum(axis=(-2, -1)) + true.sum(axis=(-2, -1)), 1)


kare_goruntuleri = np.stack(veri["kare"]).astype(np.float32)
kaviteler = np.stack(veri["kavite"]).astype(np.int64)
kare_kumeleri = veri["küme"].to_numpy()
egitimde, dogrulamada, testte = kare_kumeleri == "eğitim", kare_kumeleri == "doğrulama", kare_kumeleri == "test"
X_img, Y_cav = torch.tensor(kare_goruntuleri)[:, None], torch.tensor(kaviteler)
torch.manual_seed(2026)
unet = UNet()
epochs, batch = 15, 16
optimizer = torch.optim.Adam(unet.parameters(), lr=3e-3)
cizelge = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=3e-3, total_steps=epochs * ((int(egitimde.sum()) + batch - 1) // batch))
sira_ureteci = torch.Generator().manual_seed(2026)
X_tr, Y_tr = X_img[egitimde], Y_cav[egitimde]
best, best_state = -1.0, None
for epoch in range(epochs):
    unet.train()
    order = torch.randperm(len(X_tr), generator=sira_ureteci)
    for start in range(0, len(X_tr), batch):
        j = order[start:start + batch]
        out = unet(X_tr[j])
        loss = F.cross_entropy(out, Y_tr[j]) + soft_dice_loss(out, Y_tr[j])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        cizelge.step()
    unet.eval()
    with torch.no_grad():
        val_dice = float(dice(unet(X_img[dogrulamada]).argmax(dim=1).numpy() == 1, kaviteler[dogrulamada] == 1).mean())
    if val_dice > best:
        best, best_state, best_epoch = val_dice, copy.deepcopy(unet.state_dict()), epoch + 1
unet.load_state_dict(best_state)
unet.eval()
with torch.no_grad():
    test_prob = F.softmax(unet(X_img[testte]), dim=1)[:, 1].numpy()
cevap = round(float(dice(test_prob > 0.5, kaviteler[testte] == 1).mean()), 3)
print(int(egitimde.sum()), int(testte.sum()), sum(p.numel() for p in unet.parameters() if p.requires_grad),
      "en iyi epoch:", best_epoch, "test Dice'ı:", cevap)
teach.check_step("Adım 2.2", "cevap", globals())

**Sonuç ne gösteriyor?** Dersin çalıştırmasında 121.394 ağırlıklı U-Net, 800 eğitim karesiyle test karelerinde yaklaşık 0,924 ortalama Dice katsayısına ulaşmıştır. Eğitim bilgisayardan bilgisayara biraz değiştiği için kontrol, bu değerin 0,05 yakınındaki sonuçları kabul etmektedir.

### Adım 2.3: Görüntü ve faza göre Dice katsayısı

**Amaç.** Bu adımda, bir ekokardiyografi laboratuvarının soracağı gibi, ağın sınırları en kötü nerede çizdiği bulunur. Veri hücresi dersin U-Net'ini bir kez eğitir; bu yaklaşık üç dakika sürer.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Bir ekokardiyografi laboratuvarının soracağı gibi, bir U-Net'in sol ventrikül kavitesini hangi görüntüde ve hangi fazda en kötü çizdiğini öğrenmek istiyorum.  
> Veri: Defterde "veri" adında bir tablo var; CAMUS'un 200 test karesini, yani 50 hastanın apikal iki boşluk ve dört boşluk görüntülerinin diyastol sonu ve sistol sonu karelerini tutar. Her satır bir karedir: "hasta" sütunu hastayı, "görüntü" görüntüyü ("2CH" ya da "4CH"), "faz" ise fazı ("ED" ya da "ES") verir. "olasılık" sütununun her hücresi, 96 × 96 piksellik karenin her pikseli için bir U-Net'in verdiği sol ventrikül kavitesi olasılığını 0 ile 1 arasında bir sayı olarak tutar. "kavite" sütununun her hücresi ise uzmanın çizdiği kaviteyi aynı boyutta tutar: Kavitenin içindeki pikseller 1, dışındakiler 0'dır.  
> Görev: Her karenin olasılığını 0,5'te eşikle ve karenin Dice katsayısını uzmanın çizdiği kaviteye göre hesapla: Ortak piksellerin iki katı, iki kavitenin piksellerinin toplamına bölünür. Kareleri görüntüye ve faza göre grupla ve dört grubun, yani 2CH ED, 2CH ES, 4CH ED ve 4CH ES gruplarının kare sayısını ve ortalama Dice katsayısını gösteren küçük bir tablo yazdır.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Dört ortalamanın en düşüğünü üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Dört grubun her birinde 50 kare bulunmalı ve dört ortalamanın ortalaması 200 karenin ortalama Dice katsayısına eşit olmalı.  

In [ ]:
#@title Adım 2.3 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_p, _te = _course_unet(), frame_split == "testing"
veri = pd.DataFrame({"hasta": frame_patient[_te], "görüntü": frame_view[_te], "faz": frame_phase[_te],
                     "olasılık": list(_p), "kavite": list((masks[_te] == 1).astype(np.uint8))})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 2.3 kontrolü
teach.check_step("Adım 2.3", "cevap", globals())

In [ ]:
#@title Adım 2.3 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_p, _te = _course_unet(), frame_split == "testing"
veri = pd.DataFrame({"hasta": frame_patient[_te], "görüntü": frame_view[_te], "faz": frame_phase[_te],
                     "olasılık": list(_p), "kavite": list((masks[_te] == 1).astype(np.uint8))})
kare_basina = []
for olasilik, kav in zip(veri["olasılık"], veri["kavite"]):
    pred, true = olasilik > 0.5, kav == 1
    kare_basina.append(2 * (pred & true).sum() / max(pred.sum() + true.sum(), 1))
veri["Dice"] = kare_basina
gruplara_gore = veri.groupby(["görüntü", "faz"])["Dice"].agg(["count", "mean"]).round(3)
print(gruplara_gore)
cevap = round(float(gruplara_gore["mean"].min()), 3)
print("En düşük ortalama:", cevap)
teach.check_step("Adım 2.3", "cevap", globals())

**Sonuç ne gösteriyor?** En zayıf grubun ortalama Dice katsayısı yaklaşık 0,908 olarak bulunur; dersin çalıştırmasında bu grup, sistol sonundaki iki boşluk görüntüsüydü. ES'de kavite küçüktür; bu yüzden aynı kalınlıktaki bir sınır, kavitenin daha büyük bir kısmını oluşturur. Ayrıca apeks ve lateral duvar bu karelerde çoğu zaman daha zor görülür.

**2. durak tamamlandı.** [Ders sayfasına](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-04/ders.html#stop-2) dönün ve duraktan sonrasını okumaya devam edin.

## Durak 3: Disk yöntemiyle ejeksiyon fraksiyonu

*Ders: [3. uygulama durağı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-04/ders.html#stop-3)* · *Python yolunda aynı sonuç: Kısım 2, 8-9. bölümler*

Sınırlar, bir ekokardiyografi laboratuvarında olduğu gibi biplan disk yöntemiyle sayılara dönüşür. Bu durakta 201 numaralı hastanın hacimleri ve ejeksiyon fraksiyonu (EF) uzman sınırlarından hesaplanır, ağ 50 test hastasında bir Bland-Altman analiziyle uzmanla karşılaştırılır ve vaka, disklerin kalınlığı için başka bir kuralla yeniden ölçülür.

**Durağın sonunda:** 201 numaralı hastanın uzman sınırlarından elde edilen EF'sini (yüzde 42,1), ağ ile uzman arasındaki 50 hastalık ortalama farkı (yaklaşık −4,24 puan) ve her faz için ayrı bir kalınlıkla vakanın EF'sini (yüzde 50,3) bulacaksınız.

In [ ]:
#@title 3. durağın hazırlığı: Bu hücreyi bir kez çalıştırın (durağın verilerini yükler)
if "frames_" not in globals():
    frames_ = camus.frames(camus.patients(), size=96)
images, masks = frames_["images"], frames_["masks"]
frame_patient, frame_view, frame_phase, frame_split = frames_["patient"], frames_["view"], frames_["phase"], frames_["split"]
frame_shape = frames_["shape"]

def unet_block(c_in, c_out):
    return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
                         nn.Conv2d(c_out, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU())


class UNet(nn.Module):
    """Üç düzey aşağı, bir dar boğaz ve atlama bağlantılarıyla üç düzey yukarı."""

    def __init__(self, c=8, n_classes=2):
        super().__init__()
        self.down1, self.down2, self.down3 = unet_block(1, c), unet_block(c, 2 * c), unet_block(2 * c, 4 * c)
        self.bottom = unet_block(4 * c, 8 * c)
        self.up3, self.dec3 = nn.ConvTranspose2d(8 * c, 4 * c, 2, stride=2), unet_block(8 * c, 4 * c)
        self.up2, self.dec2 = nn.ConvTranspose2d(4 * c, 2 * c, 2, stride=2), unet_block(4 * c, 2 * c)
        self.up1, self.dec1 = nn.ConvTranspose2d(2 * c, c, 2, stride=2), unet_block(2 * c, c)
        self.out = nn.Conv2d(c, n_classes, 1)

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(F.max_pool2d(d1, 2))
        d3 = self.down3(F.max_pool2d(d2, 2))
        b = self.bottom(F.max_pool2d(d3, 2))
        u3 = self.dec3(torch.cat([self.up3(b), d3], dim=1))
        u2 = self.dec2(torch.cat([self.up2(u3), d2], dim=1))
        u1 = self.dec1(torch.cat([self.up1(u2), d1], dim=1))
        return self.out(u1)


def soft_dice_loss(logits, target):
    p = F.softmax(logits, dim=1)[:, 1]
    t = target.float()
    inter = (p * t).sum(dim=(1, 2))
    return 1 - ((2 * inter + 1) / (p.sum(dim=(1, 2)) + t.sum(dim=(1, 2)) + 1)).mean()


def dice(pred, true):
    """Her karenin Dice katsayısı: aynı boyutta iki mantıksal dizi."""
    inter = (pred & true).sum(axis=(-2, -1))
    return 2 * inter / np.maximum(pred.sum(axis=(-2, -1)) + true.sum(axis=(-2, -1)), 1)


def _course_unet():
    """Dersin U-Net'i; bir kez eğitilir (yaklaşık üç dakika) ve 200 test karesinin her pikselinde kavite olasılığını verir."""
    if "_unet_cache" in globals():
        return _unet_cache
    cavity = masks == 1
    tr, va, te = frame_split == "training", frame_split == "validation", frame_split == "testing"
    X_img = torch.tensor(images)[:, None]
    Y_cav = torch.tensor(cavity.astype(np.int64))
    torch.manual_seed(RANDOM_STATE)
    unet = UNet()
    epochs, batch = 15, 16
    optimizer = torch.optim.Adam(unet.parameters(), lr=3e-3)
    steps = epochs * ((int(tr.sum()) + batch - 1) // batch)
    schedule = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=3e-3, total_steps=steps)
    order_gen = torch.Generator().manual_seed(RANDOM_STATE)
    X_tr, Y_tr = X_img[tr], Y_cav[tr]
    best, best_state = -1.0, None
    for epoch in range(epochs):
        unet.train()
        order = torch.randperm(len(X_tr), generator=order_gen)
        for start in range(0, len(X_tr), batch):
            j = order[start:start + batch]
            out = unet(X_tr[j])
            loss = F.cross_entropy(out, Y_tr[j]) + soft_dice_loss(out, Y_tr[j])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            schedule.step()
        unet.eval()
        with torch.no_grad():
            val_dice = float(dice(unet(X_img[va]).argmax(dim=1).numpy() == 1, cavity[va]).mean())
        if val_dice > best:
            best, best_state, best_epoch = val_dice, copy.deepcopy(unet.state_dict()), epoch + 1
    unet.load_state_dict(best_state)
    unet.eval()
    with torch.no_grad():
        test_prob = F.softmax(unet(X_img[te]), dim=1)[:, 1].numpy()
    globals()["_unet_cache"] = test_prob
    return test_prob


def _course_agreement():
    """50 test hastasının uzman sınırlarından ve dersin U-Net'inin sınırlarından ölçülen hacimleri ve EF'si (bir kez hesaplanır)."""
    if "_agreement_cache" in globals():
        return _agreement_cache
    prob = _course_unet()
    outlines = {}
    for k, i in enumerate(np.where(frame_split == "testing")[0]):
        outlines.setdefault(frame_patient[i], {})[(frame_view[i], frame_phase[i])] = camus.to_original(prob[k], frame_shape[i])
    rows = []
    for name in camus.patients("testing"):
        ref, net = camus.measure(name), camus.measure(name, outlines[name])
        rows.append({"patient": name, "edv_expert": ref["edv_ml"], "edv_net": net["edv_ml"], "esv_expert": ref["esv_ml"],
                     "esv_net": net["esv_ml"], "ef_expert": ref["ef"], "ef_net": net["ef"]})
    globals()["_agreement_cache"] = pd.DataFrame(rows)
    return _agreement_cache


def _disc_rows(names):
    """Her hastanın dört uzman sınırının 20 diski; karelerin özgün boyutunda."""
    rows = []
    for name in names:
        for view in ("2CH", "4CH"):
            for phase in ("ED", "ES"):
                _, gt, spacing = camus.load(name, view, phase)
                diameters, thickness, _, _ = camus.discs(gt == 1, spacing)
                for k, d in enumerate(diameters, start=1):
                    rows.append({"hasta": name, "görüntü": view, "faz": phase, "disk": k,
                                 "çap": float(d), "uzun eksen": float(thickness * 20)})
    return pd.DataFrame(rows)

print(len(camus.patients("testing")), "test hastası")
print("3. durak hazır.")

### Adım 3.1: Disk yöntemiyle 201 numaralı hasta

**Amaç.** Bu adımda haftanın vakası, CAMUS'un referansında olduğu gibi uzmanın sınırlarından ölçülür.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: CAMUS'taki 201 numaralı hastanın diyastol sonu ve sistol sonu hacimlerini ve ejeksiyon fraksiyonunu, CAMUS'un referansındaki gibi uzman sınırlarından biplan disk yöntemiyle hesaplamak istiyorum.  
> Veri: Defterde "veri" adında bir tablo var; CAMUS'taki 201 numaralı hastada sol ventrikül kavitesinin dört uzman sınırına uygulanan disk yönteminin disklerini tutar. Bu sınırlar apikal iki boşluk (2CH) ve dört boşluk (4CH) görüntülerinin diyastol sonu (ED) ve sistol sonu (ES) karelerindedir. Her sınır, mitral halkadan apekse uzanan uzun eksenine dik 20 diske bölünmüştür. Her satır bir disktir: "görüntü" ve "faz" sütunları sınırı, "disk" diskin 1'den 20'ye kadar numarasını, "çap" diskin çapını mm olarak, "uzun eksen" ise sınırın uzun ekseninin uzunluğunu mm olarak verir; uzun eksen, sınırın 20 diskinde de aynıdır. Çaplar ve uzunluklar, karelerin özgün boyutunda ve pikselin gerçek büyüklüğüyle ölçülmüştür.  
> Görev: Her hacmi biplan disk yöntemiyle, V = π/4 × Σ (a_i × b_i) × h formülüyle hesapla; burada a_i, 2CH sınırındaki i. diskin çapı, b_i aynı fazın 4CH sınırındaki aynı diskin çapı, h ise bir diskin kalınlığıdır. CAMUS'un referansındaki gibi iki faz için de tek bir kalınlık kullan; bu kalınlık, dört sınırın uzun eksenlerinden en uzununun 20'ye bölümüdür. mm³'ü mL'ye çevirmek için 1.000'e böl, her hacmi tam sayı mL'ye yuvarla ve EF'yi yuvarlanmış hacimlerden (EDV − ESV)/EDV × 100 formülüyle hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: EDV ve ESV değerlerini mL olarak yazdır; EF'yi yüzde olarak bir ondalık basamakla "cevap" adıyla sakla.  
> Test: EDV, ESV'den büyük olmalı ve 50 ile 300 mL arasında bulunmalı.  

In [ ]:
#@title Adım 3.1 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _disc_rows(["patient0201"]).drop(columns="hasta")
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 3.1 kontrolü
teach.check_step("Adım 3.1", "cevap", globals())

In [ ]:
#@title Adım 3.1 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = _disc_rows(["patient0201"]).drop(columns="hasta")


def hacim(faz_adi, kalinlik):
    parca = veri[veri["faz"] == faz_adi].sort_values("disk")
    a = parca[parca["görüntü"] == "2CH"]["çap"].to_numpy()
    b = parca[parca["görüntü"] == "4CH"]["çap"].to_numpy()
    return round(np.pi / 4 * float(np.sum(a * b)) * kalinlik / 1000)


kalinlik = veri["uzun eksen"].max() / 20
edv, esv = hacim("ED", kalinlik), hacim("ES", kalinlik)
cevap = round(100 * (edv - esv) / edv, 1)
print("EDV:", edv, "mL · ESV:", esv, "mL · EF:", cevap, "%")
teach.check_step("Adım 3.1", "cevap", globals())

**Sonuç ne gösteriyor?** Uzman sınırları 159 mL'lik bir diyastol sonu hacmi ve 92 mL'lik bir sistol sonu hacmi verir; EF yüzde 42,1 olarak hesaplanır. Bu değer hafif azalmış bir ejeksiyon fraksiyonuna karşılık gelir; haftanın vakası, ağın da bu değere ulaşıp ulaşamayacağını sormaktadır.

### Adım 3.2: Ağın uzmanla uyumu

**Amaç.** Bu adımda otomatik EF, bir laboratuvarın kullanmadan önce yapacağı gibi 50 test hastasında uzmanınkiyle karşılaştırılır. Dersin U-Net'i henüz bellekte değilse veri hücresi önce onu eğitir; bu yaklaşık üç dakika sürer.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Otomatik bir ejeksiyon fraksiyonu kullanılmadan önce, bir laboratuvarın yapacağı gibi, onu CAMUS'un 50 test hastasında bir Bland-Altman analiziyle uzmanınkiyle karşılaştırmak istiyorum.  
> Veri: Defterde "veri" adında bir tablo var; CAMUS'un 50 test hastasını tutar. Her satır bir hastadır: "hasta" sütunu hastayı, "uzman EF" uzmanın çizdiği sınırlardan, "ağ EF" ise bir U-Net'in çizdiği sınırlardan elde edilen ejeksiyon fraksiyonunu verir; ikisi de yüzde olarak ve biplan disk yöntemiyle ölçülmüştür.  
> Görev: Her hasta için EF farkını (ağ eksi uzman) hesapla; ardından farkların ortalamasını, standart sapmasını ve uyum sınırlarını, yani ortalama ± 1,96 standart sapmayı bul. Bir Bland-Altman grafiği çiz: Farkı iki EF'nin ortalamasına karşı göster; ortalama fark ve iki uyum sınırı için yatay çizgiler ekle.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Standart sapmayı ve iki uyum sınırını yazdır; ortalama farkı puan cinsinden iki ondalık basamakla "cevap" adıyla sakla.  
> Test: 50 hasta bulunmalı ve patient0201 hastasının uzman EF'si yüzde 42,1 olmalı.  

In [ ]:
#@title Adım 3.2 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_a = _course_agreement()
veri = pd.DataFrame({"hasta": _a["patient"], "uzman EF": _a["ef_expert"], "ağ EF": _a["ef_net"]})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 3.2 kontrolü
teach.check_step("Adım 3.2", "cevap", globals())

In [ ]:
#@title Adım 3.2 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_a = _course_agreement()
veri = pd.DataFrame({"hasta": _a["patient"], "uzman EF": _a["ef_expert"], "ağ EF": _a["ef_net"]})
fark = veri["ağ EF"] - veri["uzman EF"]
ort_ef = (veri["ağ EF"] + veri["uzman EF"]) / 2
bias, sd = float(fark.mean()), float(fark.std(ddof=1))
low, high = bias - 1.96 * sd, bias + 1.96 * sd
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(ort_ef, fark, s=18, color="#1F5F8B")
for value, style in ((bias, "-"), (low, "--"), (high, "--")):
    ax.axhline(value, color="#9B1B1B", ls=style, lw=1)
ax.set_xlabel("İki EF'nin ortalaması (%)")
ax.set_ylabel("Ağ eksi uzman (puan)")
plt.show()
cevap = round(bias, 2)
print(len(veri), "hasta; patient0201 için uzmanın EF'si:",
      float(veri.set_index("hasta").loc["patient0201", "uzman EF"]))
print("Ortalama fark:", cevap, "SS:", round(sd, 2), "sınırlar:", round(low, 1), round(high, 1))
teach.check_step("Adım 3.2", "cevap", globals())

**Sonuç ne gösteriyor?** Dersin çalıştırmasında ağ ile uzman arasındaki ortalama fark yaklaşık −4,24 puandır; yani ağ EF'yi ortalama olarak uzmandan düşük ölçmüştür. Uyum sınırları ise ortalama farkın iki yanında yaklaşık 16 puan uzaklıktadır. Bir laboratuvar, hangi genişliği kabul edeceğine testten önce karar verir; korelasyon tek başına iki yöntemin birbirinin yerine geçip geçemeyeceğini söyleyemez.

### Adım 3.3: Disklerin kalınlığı için başka bir kural

**Amaç.** Bu adımda bir sınırın değil, yazılımın bir kuralının vakanın EF'sini ne kadar değiştirdiği görülür.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Bir sınırın değil, disklerin kalınlığı için seçilen kuralın 201 numaralı hastanın ejeksiyon fraksiyonunu ne kadar değiştirdiğini görmek istiyorum.  
> Veri: Defterde "veri" adında bir tablo var; CAMUS'taki 201 numaralı hastada sol ventrikül kavitesinin dört uzman sınırına uygulanan disk yönteminin disklerini tutar. Bu sınırlar apikal iki boşluk (2CH) ve dört boşluk (4CH) görüntülerinin diyastol sonu (ED) ve sistol sonu (ES) karelerindedir. Her sınır, mitral halkadan apekse uzanan uzun eksenine dik 20 diske bölünmüştür. Her satır bir disktir: "görüntü" ve "faz" sütunları sınırı, "disk" diskin 1'den 20'ye kadar numarasını, "çap" diskin çapını mm olarak, "uzun eksen" ise sınırın uzun ekseninin uzunluğunu mm olarak verir; uzun eksen, sınırın 20 diskinde de aynıdır. Çaplar ve uzunluklar, karelerin özgün boyutunda ve pikselin gerçek büyüklüğüyle ölçülmüştür.  
> Görev: Her hacmi biplan disk yöntemiyle, V = π/4 × Σ (a_i × b_i) × h formülüyle hesapla; burada a_i, 2CH sınırındaki i. diskin çapı, b_i aynı fazın 4CH sınırındaki aynı diskin çapı, h ise bir diskin kalınlığıdır. Bu kez her faz için ayrı bir kalınlık kullan; bu kalınlık, o fazın iki sınırının uzun eksenlerinden daha uzun olanının 20'ye bölümüdür. mm³'ü mL'ye çevirmek için 1.000'e böl, her hacmi tam sayı mL'ye yuvarla ve EF'yi yuvarlanmış hacimlerden (EDV − ESV)/EDV × 100 formülüyle hesapla. Karşılaştırma için EDV'yi, dört sınırın uzun eksenlerinden en uzunundan gelen tek bir kalınlıkla da hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: EDV ve ESV değerlerini mL olarak yazdır; EF'yi yüzde olarak bir ondalık basamakla "cevap" adıyla sakla.  
> Test: İki EDV eşit olmalı; çünkü diyastol sonunda iki kural da aynı uzun ekseni kullanır.  

In [ ]:
#@title Adım 3.3 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _disc_rows(["patient0201"]).drop(columns="hasta")
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 3.3 kontrolü
teach.check_step("Adım 3.3", "cevap", globals())

In [ ]:
#@title Adım 3.3 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = _disc_rows(["patient0201"]).drop(columns="hasta")


def hacim(faz_adi, kalinlik):
    parca = veri[veri["faz"] == faz_adi].sort_values("disk")
    a = parca[parca["görüntü"] == "2CH"]["çap"].to_numpy()
    b = parca[parca["görüntü"] == "4CH"]["çap"].to_numpy()
    return round(np.pi / 4 * float(np.sum(a * b)) * kalinlik / 1000)


eksenler = veri.groupby("faz")["uzun eksen"].max()
edv, esv = hacim("ED", eksenler["ED"] / 20), hacim("ES", eksenler["ES"] / 20)
cevap = round(100 * (edv - esv) / edv, 1)
print("EDV:", edv, "mL · ESV:", esv, "mL · EF:", cevap, "% · tek kalınlıkla EDV:",
      hacim("ED", veri["uzun eksen"].max() / 20), "mL")
teach.check_step("Adım 3.3", "cevap", globals())

**Sonuç ne gösteriyor?** Her faz için ayrı bir kalınlıkla sistol sonu hacmi 92 mL'den 79 mL'ye düşer ve EF yükselerek yüzde 50,3 olur; oysa sınırların tek bir pikseli bile değişmemiştir. Otomatik bir EF bir laboratuvarla karşılaştırılmadan önce, laboratuvar yazılımının kullandığı kural bilinmelidir.

**3. durak tamamlandı.** [Ders sayfasına](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-04/ders.html#stop-3) dönün ve duraktan sonrasını okumaya devam edin.

## Prompt yolunda haftalık proje

*Haftanın projesi: [4. haftanın README sayfası, Haftalık proje](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-04/README.md#haftal%C4%B1k-proje)*

Haftalık proje, her adım için bir prompt olmak üzere aşağıdaki promptlarla da yapılabilir. Önce hazırlık hücresini bir kez çalıştırın; hücre projenin verilerini yükler. Ardından her adım, duraklardaki adımlar gibi ilerler. Tek fark şudur: Projede kontrol hücresi ve hazır kod yoktur, çünkü sonuçlar rapora girer. Bir promptun testi sağlanmadan sonraki prompta geçilmez ve her prompt YZ kullanım günlüğüne yazılır.

In [ ]:
#@title Projenin hazırlığı: Bu hücreyi bir kez çalıştırın (projenin verilerini yükler)
if "frames_" not in globals():
    frames_ = camus.frames(camus.patients(), size=96)
images, masks = frames_["images"], frames_["masks"]
frame_patient, frame_view, frame_phase, frame_split = frames_["patient"], frames_["view"], frames_["phase"], frames_["split"]
frame_shape = frames_["shape"]

def unet_block(c_in, c_out):
    return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
                         nn.Conv2d(c_out, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU())


class UNet(nn.Module):
    """Üç düzey aşağı, bir dar boğaz ve atlama bağlantılarıyla üç düzey yukarı."""

    def __init__(self, c=8, n_classes=2):
        super().__init__()
        self.down1, self.down2, self.down3 = unet_block(1, c), unet_block(c, 2 * c), unet_block(2 * c, 4 * c)
        self.bottom = unet_block(4 * c, 8 * c)
        self.up3, self.dec3 = nn.ConvTranspose2d(8 * c, 4 * c, 2, stride=2), unet_block(8 * c, 4 * c)
        self.up2, self.dec2 = nn.ConvTranspose2d(4 * c, 2 * c, 2, stride=2), unet_block(4 * c, 2 * c)
        self.up1, self.dec1 = nn.ConvTranspose2d(2 * c, c, 2, stride=2), unet_block(2 * c, c)
        self.out = nn.Conv2d(c, n_classes, 1)

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(F.max_pool2d(d1, 2))
        d3 = self.down3(F.max_pool2d(d2, 2))
        b = self.bottom(F.max_pool2d(d3, 2))
        u3 = self.dec3(torch.cat([self.up3(b), d3], dim=1))
        u2 = self.dec2(torch.cat([self.up2(u3), d2], dim=1))
        u1 = self.dec1(torch.cat([self.up1(u2), d1], dim=1))
        return self.out(u1)


def soft_dice_loss(logits, target):
    p = F.softmax(logits, dim=1)[:, 1]
    t = target.float()
    inter = (p * t).sum(dim=(1, 2))
    return 1 - ((2 * inter + 1) / (p.sum(dim=(1, 2)) + t.sum(dim=(1, 2)) + 1)).mean()


def dice(pred, true):
    """Her karenin Dice katsayısı: aynı boyutta iki mantıksal dizi."""
    inter = (pred & true).sum(axis=(-2, -1))
    return 2 * inter / np.maximum(pred.sum(axis=(-2, -1)) + true.sum(axis=(-2, -1)), 1)


def _course_unet():
    """Dersin U-Net'i; bir kez eğitilir (yaklaşık üç dakika) ve 200 test karesinin her pikselinde kavite olasılığını verir."""
    if "_unet_cache" in globals():
        return _unet_cache
    cavity = masks == 1
    tr, va, te = frame_split == "training", frame_split == "validation", frame_split == "testing"
    X_img = torch.tensor(images)[:, None]
    Y_cav = torch.tensor(cavity.astype(np.int64))
    torch.manual_seed(RANDOM_STATE)
    unet = UNet()
    epochs, batch = 15, 16
    optimizer = torch.optim.Adam(unet.parameters(), lr=3e-3)
    steps = epochs * ((int(tr.sum()) + batch - 1) // batch)
    schedule = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=3e-3, total_steps=steps)
    order_gen = torch.Generator().manual_seed(RANDOM_STATE)
    X_tr, Y_tr = X_img[tr], Y_cav[tr]
    best, best_state = -1.0, None
    for epoch in range(epochs):
        unet.train()
        order = torch.randperm(len(X_tr), generator=order_gen)
        for start in range(0, len(X_tr), batch):
            j = order[start:start + batch]
            out = unet(X_tr[j])
            loss = F.cross_entropy(out, Y_tr[j]) + soft_dice_loss(out, Y_tr[j])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            schedule.step()
        unet.eval()
        with torch.no_grad():
            val_dice = float(dice(unet(X_img[va]).argmax(dim=1).numpy() == 1, cavity[va]).mean())
        if val_dice > best:
            best, best_state, best_epoch = val_dice, copy.deepcopy(unet.state_dict()), epoch + 1
    unet.load_state_dict(best_state)
    unet.eval()
    with torch.no_grad():
        test_prob = F.softmax(unet(X_img[te]), dim=1)[:, 1].numpy()
    globals()["_unet_cache"] = test_prob
    return test_prob


def _course_agreement():
    """50 test hastasının uzman sınırlarından ve dersin U-Net'inin sınırlarından ölçülen hacimleri ve EF'si (bir kez hesaplanır)."""
    if "_agreement_cache" in globals():
        return _agreement_cache
    prob = _course_unet()
    outlines = {}
    for k, i in enumerate(np.where(frame_split == "testing")[0]):
        outlines.setdefault(frame_patient[i], {})[(frame_view[i], frame_phase[i])] = camus.to_original(prob[k], frame_shape[i])
    rows = []
    for name in camus.patients("testing"):
        ref, net = camus.measure(name), camus.measure(name, outlines[name])
        rows.append({"patient": name, "edv_expert": ref["edv_ml"], "edv_net": net["edv_ml"], "esv_expert": ref["esv_ml"],
                     "esv_net": net["esv_ml"], "ef_expert": ref["ef"], "ef_net": net["ef"]})
    globals()["_agreement_cache"] = pd.DataFrame(rows)
    return _agreement_cache
_FSETS = {"training": "eğitim", "validation": "doğrulama", "testing": "test"}

_QUALITY = {"Good": "iyi", "Medium": "orta", "Poor": "kötü"}
_quality = {(r["patient"], v): _QUALITY.get(r["quality_" + v.lower()], r["quality_" + v.lower()])
            for _, r in camus.table().iterrows() for v in ("2CH", "4CH")}
print("Proje hazır.")

### Proje adımı 1: U-Net'i eğitmek ve sınırlarını ölçmek

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: CAMUS ekokardiyogramlarında sol ventrikül kavitesinin sınırını çizen bir U-Net eğitmek ve ağın sınırları nerede daha kötü çizdiğini görüntüye, faza ve görüntü kalitesine göre görmek istiyorum.  
> Veri: Defterde "veri" adında bir tablo var; CAMUS'un 1.200 karesini, yani 300 hastanın apikal iki boşluk ve dört boşluk görüntülerinin diyastol sonu ve sistol sonu karelerini tutar. Her satır bir karedir: "hasta" sütunu hastayı, "görüntü" görüntüyü ("2CH" ya da "4CH"), "faz" fazı ("ED" ya da "ES"), "küme" hastanın kümesini ("eğitim", 200 hasta; "doğrulama", 50 hasta; "test", 50 hasta), "görüntü kalitesi" ise yazarların hastanın bu görüntüsüne verdiği kaliteyi ("iyi", "orta" ya da "kötü") verir. "kare" sütununun her hücresi kareyi değerleri 0 ile 1 arasında olan 96 × 96 piksellik bir görüntü olarak, "kavite" sütununun her hücresi ise uzmanın çizdiği sol ventrikül kavitesini aynı boyutta tutar: Kavitenin içindeki pikseller 1, dışındakiler 0'dır.  
> Görev: Her piksele, biri kavite, biri kavite dışı için olmak üzere iki skor veren küçük bir U-Net kur. Ağda atlama bağlantılarıyla üç düzey aşağı, bir dar boğaz ve üç düzey yukarı bulunur. Her düzeyde görüntünün boyutunu koruyan 3 × 3 çekirdekli iki evrişim vardır ve her evrişimin ardından batch normalizasyonu ile ReLU gelir. Aşağı inerken 2 × 2 maksimum havuzlama görüntüyü yarıya indirir; yukarı çıkarken 2 × 2'lik ters evrişim görüntüyü iki kat büyütür ve sonuç, aşağı inerken aynı düzeyde elde edilen çıktıyla birleştirilir. İlk düzeyde 8 kanal vardır ve kanal sayısı aşağı inilen her düzeyde iki katına çıkarak dar boğazda 64'e ulaşır. Son bir 1 × 1 evrişim iki skoru verir. Ağı eğitim karelerinde 15 epoch boyunca, 16 karelik karıştırılmış gruplarla eğit. Kayıp, çapraz entropi ile yumuşak Dice kaybının toplamıdır; yumuşak Dice kaybı, kavite olasılıklarından hesaplanan Dice katsayısının 1'den çıkarılmasıyla bulunur. En yüksek değeri 0,003 olan tek döngülü bir öğrenme hızı kullan. Her epoch'un sonunda doğrulama karelerinin ortalama Dice katsayısını hesapla ve bu değerin en yüksek olduğu epoch'un ağırlıklarını sakla. Rastgele seçimler her çalıştırmada aynı olsun; bunun için başlangıç değeri olarak 2026 sayısını kullan. Ardından test karelerindeki kavite olasılığını 0,5'te eşikle ve her test karesinin Dice katsayısını uzmanın çizdiği kaviteye göre hesapla. Ortalama Dice katsayısını görüntüye ve faza göre ve görüntü kalitesine göre bildir.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Görüntü ve faza göre ortalama Dice katsayısını gösteren bir tablo ile görüntü kalitesine göre ikinci bir tablo göster; 200 test karesinin ortalama Dice katsayısını üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Görüntü ve fazın dört grubunun her birinde 50 test karesi bulunmalı ve 200 test karesinin ortalama Dice katsayısı 0,85 ile 0,97 arasında olmalı.  

In [ ]:
#@title Proje adımı 1 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_FSETS = {"training": "eğitim", "validation": "doğrulama", "testing": "test"}
veri = pd.DataFrame({"hasta": frame_patient, "görüntü": frame_view, "faz": frame_phase,
                     "küme": [_FSETS[s] for s in frame_split],
                     "görüntü kalitesi": [_quality[(p, v)] for p, v in zip(frame_patient, frame_view)],
                     "kare": list(images), "kavite": list((masks == 1).astype(np.uint8))})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

### Proje adımı 2: Hacimler, EF ve Bland-Altman

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Otomatik hacimler ve otomatik bir ejeksiyon fraksiyonu kullanılmadan önce üçünü de CAMUS'un 50 test hastasında uzmanınkilerle karşılaştırmak istiyorum.  
> Veri: Defterde "veri" adında bir tablo var; CAMUS'un 50 test hastasını tutar. Her satır bir hastadır: "hasta" sütunu hastayı verir; "uzman EDV" ve "ağ EDV" diyastol sonu hacmini, "uzman ESV" ve "ağ ESV" sistol sonu hacmini mL olarak, "uzman EF" ve "ağ EF" ise ejeksiyon fraksiyonunu yüzde olarak verir. Uzmanın değerleri uzmanın çizdiği sınırlardan, ağın değerleri ise bir U-Net'in çizdiği sınırlardan gelir ve hepsi biplan disk yöntemiyle ölçülmüştür.  
> Görev: Üç ölçümün, yani EDV, ESV ve EF'nin her biri için her hastadaki farkı (ağ eksi uzman) hesapla; ardından ortalama farkı, farkların standart sapmasını ve uyum sınırlarını, yani ortalama ± 1,96 standart sapmayı bul. Her ölçüm için bir Bland-Altman grafiği çiz: Farkı iki değerin ortalamasına karşı göster; ortalama fark ve iki uyum sınırı için yatay çizgiler ekle.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Her ölçüm için bir satır ve ortalama fark, standart sapma, alt sınır ve üst sınır sütunları bulunan bir tablo ile üç Bland-Altman grafiğini göster; EF'nin ortalama farkını iki ondalık basamakla "cevap" adıyla sakla.  
> Test: 50 hasta bulunmalı ve patient0201 hastasının uzman EF'si yüzde 42,1 olmalı.  

In [ ]:
#@title Proje adımı 2 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_a = _course_agreement()
veri = pd.DataFrame({"hasta": _a["patient"],
                     "uzman EDV": _a["edv_expert"], "ağ EDV": _a["edv_net"],
                     "uzman ESV": _a["esv_expert"], "ağ ESV": _a["esv_net"],
                     "uzman EF": _a["ef_expert"], "ağ EF": _a["ef_net"]})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

### Proje adımı 3: EF aralıkları

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: 50 test hastasının kaçında ağın ve uzmanın ejeksiyon fraksiyonunu aynı klinik aralığa yerleştirdiğini öğrenmek istiyorum.  
> Veri: Defterde "veri" adında bir tablo var; CAMUS'un 50 test hastasını tutar. Her satır bir hastadır: "hasta" sütunu hastayı verir; "uzman EDV" ve "ağ EDV" diyastol sonu hacmini, "uzman ESV" ve "ağ ESV" sistol sonu hacmini mL olarak, "uzman EF" ve "ağ EF" ise ejeksiyon fraksiyonunu yüzde olarak verir. Uzmanın değerleri uzmanın çizdiği sınırlardan, ağın değerleri ise bir U-Net'in çizdiği sınırlardan gelir ve hepsi biplan disk yöntemiyle ölçülmüştür.  
> Görev: Uzmanın ve ağın her EF değerini üç aralıktan birine yerleştir. Birinci aralık yüzde 40 ve altını, ikinci aralık yüzde 40'ın üzerini ve yüzde 50'nin altını, üçüncü aralık ise yüzde 50 ve üzerini kapsar. Satırlarda uzmanın, sütunlarda ağın aralığı aralıkların sırasıyla yer alacak biçimde bir çapraz tablo oluştur ve tablonun köşegenindeki hastaları say.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Çapraz tabloyu göster ve köşegenindeki hasta sayısını "cevap" adıyla sakla.  
> Test: Çapraz tablonun toplamı 50 hasta olmalı.  

In [ ]:
#@title Proje adımı 3 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_a = _course_agreement()
veri = pd.DataFrame({"hasta": _a["patient"],
                     "uzman EDV": _a["edv_expert"], "ağ EDV": _a["edv_net"],
                     "uzman ESV": _a["esv_expert"], "ağ ESV": _a["esv_net"],
                     "uzman EF": _a["ef_expert"], "ağ EF": _a["ef_net"]})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

### Proje adımı 4: En büyük hatalar

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Otomatik ejeksiyon fraksiyonu uzmanınkinden en çok ayrılan test hastalarında, ağın nerede yanıldığını değerlendirmek için kareleri ve iki sınırı görmek istiyorum.  
> Veri: Defterde "veri" adında bir tablo var; CAMUS'un 50 test hastasının her birinden dört kare olmak üzere 200 test karesini tutar. Her satır bir karedir: "hasta" sütunu hastayı, "görüntü" görüntüyü ("2CH" ya da "4CH"), "faz" fazı ("ED" ya da "ES"), "görüntü kalitesi" ise görüntünün kalitesini ("iyi", "orta" ya da "kötü") verir. "kare" sütununun her hücresi kareyi 96 × 96 piksellik bir görüntü olarak tutar; "uzman kavitesi" ve "ağ kavitesi" sütunları ise uzmanın ve bir U-Net'in çizdiği sol ventrikül kavitesini aynı boyutta tutar: Kavitenin içindeki pikseller 1, dışındakiler 0'dır. "uzman EF" ve "ağ EF" sütunları hastanın ejeksiyon fraksiyonunu yüzde olarak verir; bu değerler hastanın dört satırında da aynıdır.  
> Görev: İki EF arasındaki mutlak farkı en büyük olan üç hastayı bul. Her biri için dört kareyi, uzman kavitesinin ve ağ kavitesinin sınırları kare üzerinde iki ayrı renkle çizilmiş olarak göster; başlığa iki EF'yi ve görüntü kalitesini yaz.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Üç hastanın her biri için dört kareden oluşan birer şekil göster; EF'deki en büyük mutlak farkı puan cinsinden bir ondalık basamakla "cevap" adıyla sakla.  
> Test: Gösterilen üç hasta, EF'deki mutlak farkı en büyük olan üç hasta olmalı ve her şekilde dört kare bulunmalı.  

In [ ]:
#@title Proje adımı 4 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_a = _course_agreement().set_index("patient")
_p, _te = _course_unet(), np.where(frame_split == "testing")[0]
veri = pd.DataFrame({"hasta": frame_patient[_te], "görüntü": frame_view[_te], "faz": frame_phase[_te],
                     "görüntü kalitesi": [_quality[(frame_patient[i], frame_view[i])] for i in _te],
                     "kare": list(images[_te]), "uzman kavitesi": list((masks[_te] == 1).astype(np.uint8)),
                     "ağ kavitesi": list((_p > 0.5).astype(np.uint8)),
                     "uzman EF": [_a.loc[frame_patient[i], "ef_expert"] for i in _te],
                     "ağ EF": [_a.loc[frame_patient[i], "ef_net"] for i in _te]})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

<a id="part-2"></a>

# Kısım 2: Python yolu

**Kimler için?** Bu kısım Python yazmayı öğrenmek isteyen katılımcılar içindir. Kısım 1'i çalışmış olmanız gerekmez: 0. bölümden sonra doğrudan buradan başlayın.

**Nasıl ilerler?** Hücreleri yukarıdan aşağıya Shift+Enter ile çalıştırın. Baştaki *Python adımı*, haftanın gerektirdiği dil bilgisini öğretir. Numaralı bölümlerdeki alıştırmalar hemen *doğru* ya da *henüz değil* yazar; örnek çözümler katlanmış hücrelerde durur ve ancak denedikten sonra açılmalıdır. *Gözlemle ve düşün* hücreleri açık uçlu sorular sorar; bu soruların çözüm hücresi yoktur. *Python notu* hücreleri, yeni bir yapının ilk kullanıldığı yerde dili açıklar. *İleri düzey (isteğe bağlı)* olarak işaretlenen hücreler, programlama bilenler için ek çalışmalardır. Bu kısma ortadan başlayan katılımcı, o bölümün başındaki *Telafi* hücresini çalıştırır. Bazı alıştırma hücreleri `veri` adlı bir tabloyu hazırlayan bir satırla başlar. Bu tablo, etkileşimli labın prompt atölyesindeki alıştırma promptları için hazırlanmıştır; Python yolunda alıştırma, metninde adı geçen tablolarla çözülebilir.

## Python adımı 4: Eğitilmiş modelleri yeniden kullanmak

Bu hafta kod, daha önce yapılmış işleri yeniden kullanmaktadır: Bir milyondan fazla fotoğrafla eğitilmiş bir ağı ve ekokardiyogramlar üzerine çizilmiş uzman sınırlarını. Bu adımda CAMUS'un bir karesi sınırıyla birlikte okunur, sınır doğru ve yanlış değerlerinden oluşan bir diziye dönüştürülür ve sol ventrikülün alanı santimetrekare cinsinden ölçülür. Ardından EfficientNet-B0'ın ImageNet ağırlıkları yüklenir, ağın katmanları dondurulur ve hâlâ eğitilecek ağırlıklar sayılır.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
img, gt, spacing = camus.load("patient0201", "4CH", "ED")
print(img.shape, gt.shape, spacing)
print("sınırdaki etiketler:", np.unique(gt))
fig, ax = plt.subplots(figsize=(4.2, 4.2))
ax.imshow(img, cmap="gray")
ax.contour(gt == 1, levels=[0.5], colors="#E5797B", linewidths=1.2)
ax.set_title("201 numaralı hasta, dört boşluk görüntüsü, diyastol sonu", fontsize=9)
ax.axis("off")
plt.show()

**Python notu.** `camus.load` üç şey döndürür; `img, gt, spacing = ...` satırı bunların her birine tek satırda bir ad verir. Buna *açma* (unpacking) denir. `img` karenin gri değerlerini, `gt` her piksel için bir etiketi (0 arka plan, 1 sol ventrikül kavitesi, 2 miyokard, 3 sol atriyum), `spacing` ise bir pikselin mm cinsinden boyutunu (yükseklik, genişlik) tutar. `np.unique(gt)`, bir dizideki farklı değerlerin her birini bir kez listeler. Kare, apeks üstte olacak biçimde alışılmış yönde gösterilmektedir. CAMUS görüntüleri: CAMUS, CC BY-NC-SA 4.0.

In [ ]:
# Python alıştırması: bir maske ve onu ölçen bir fonksiyon
cavity = gt == 1                      # kavitenin içinde True, dışında False
print(cavity.dtype, cavity.sum(), "kavitedeki piksel")


def area_cm2(mask, spacing):
    """Bir maskenin cm² cinsinden alanı: piksel sayısı × bir pikselin alanı."""
    return float(mask.sum() * spacing[0] * spacing[1] / 100)


print("Kavitenin alanı:", round(area_cm2(cavity, spacing), 1), "cm²")

**Python notu.** `gt == 1` her pikseli 1 ile karşılaştırır ve `True` ile `False` değerlerinden oluşan bir dizi, yani bir *maske* verir. `.sum()` `True` değerlerini sayar; çünkü Python `True` değerini 1 olarak sayar. 0,308 × 0,308 mm'lik bir piksel 0,095 mm²'lik bir alan kaplar; 100 mm² ise 1 cm² eder. `def area_cm2(mask, spacing):` satırı iki değer alan ve `return` ile bir değer döndüren bir fonksiyon tanımlar; altındaki üç tırnak arasındaki metin fonksiyonu tarif eder.

In [ ]:
# Python alıştırması: eğitilmiş bir ağ, dondurulmuş hâlde
net = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
print("Bütün ağırlıklar:", sum(p.numel() for p in net.parameters()))
print(net.classifier)
for p in net.parameters():
    p.requires_grad = False                                     # donduruldu
net.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(1280, 3))  # üç grup için yeni bir son katman
print("Hâlâ eğitilen ağırlıklar:", sum(p.numel() for p in net.parameters() if p.requires_grad))

**Python notu.** `efficientnet_b0(weights=...)` ağı kurar ve ImageNet üzerinde öğrenilmiş ağırlıkları yükler; ağırlıklar 0. bölümde indirilmiştir. Bir PyTorch ağının her ağırlığının `requires_grad` adlı bir özelliği vardır. Bu özellik `False` olduğunda autograd o ağırlık için gradyan hesaplamaz ve ağırlık eğitim sırasında değişmez; böylece katman *dondurulmuş* olur. Dondurmadan sonra oluşturulan yeni bir katman `requires_grad = True` ile başlar. `sum(... for p in ... if ...)` yalnızca koşulu sağlayan elemanları toplar.

**Hızlı kontrol.** 1.280 girdisi ve 3 çıktısı olan yeni bir son katmanla donmuş bir ağ eğitilmektedir. Hangi ağırlıklar değişir?

<details><summary>Cevap</summary>

Yalnızca yeni son katmanın 1.280 × 3 ağırlığı ve 3 sabit terimi, yani 3.843 sayı değişir; ImageNet üzerinde öğrenilmiş dört milyon ağırlık olduğu gibi kalır.

</details>

**Alıştırma 4.1.** 201 numaralı hastanın dört boşluk diyastol sonu karesinde kaç piksel sol ventrikül kavitesine aittir?

In [ ]:
veri = alistirma_tablosu("4.1")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma 4.1", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = int((gt == 1).sum())
teach.check("Alıştırma 4.1", cevap)

**Alıştırma 4.2.** 201 numaralı hastanın dört boşluk sistol sonu karesinde sol ventrikül kavitesinin alanı kaç cm²'dir? Sonucu bir ondalık basamağa yuvarlayın.

In [ ]:
veri = alistirma_tablosu("4.2")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma 4.2", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
img_es, gt_es, spacing_es = camus.load("patient0201", "4CH", "ES")
cevap = round(area_cm2(gt_es == 1, spacing_es), 1)
teach.check("Alıştırma 4.2", cevap)

**Alıştırma 4.3.** `net` içinde EfficientNet-B0'ın bütün ağırlıkları dondurulmuştur ve yeni son katmanın üç çıktısı vardır. Kaç ağırlık hâlâ eğitilmektedir?

In [ ]:
cevap = None   # cevabınız
teach.check("Alıştırma 4.3", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = sum(p.numel() for p in net.parameters() if p.requires_grad)
teach.check("Alıştırma 4.3", cevap)

## 1. Bir ağın fotoğraflardan öğrendikleri

*Ders: [Transfer öğrenme: Bir ağın fotoğraflardan öğrendikleri](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-04/ders.html#transfer-learning-what-a-network-learned-from-photographs)*

EfficientNet-B0, ImageNet'in yaklaşık 1,2 milyon fotoğrafından 1.000 çeşit nesneyi adlandırmayı öğrenmiştir. İlk katmanı, üç renk kanalı üzerinde çalışan küçük filtrelerden oluşur; daha derindeki katmanlar bu filtrelerin yanıtlarını birleştirerek dokuları ve nesne parçalarını tanır. Bir sine kardiyak manyetik rezonans kesitinin renkleri yoktur. Bu yüzden eğitimde diyastol sonu karesi, sistol sonu karesi ve ikisinin ortalaması üç kanala yerleştirilir ve bu kanallar ImageNet fotoğraflarıyla aynı biçimde normalleştirilir.

**Formüllerden koda.** $c$ kanalının normalleştirmesi $x' = (x - \mu_c)/\sigma_c$ ile yapılır; burada $\mu = (0{,}485;\ 0{,}456;\ 0{,}406)$ ve $\sigma = (0{,}229;\ 0{,}224;\ 0{,}225)$'tir: `(x3 - IMAGENET_MEAN) / IMAGENET_SD`. `F.interpolate(..., size=(128, 128))` 64 × 64 piksellik kesitleri büyütür. `net.features[0][0].weight`, ilk katmanın her biri 3 kanal üzerinde 3 × 3 boyutunda olan 32 filtresini tutar.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 1. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB04_transfer_ogrenme.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-04/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(1)

In [ ]:
# İlk katmanın 32 filtresi, küçük renkli görüntüler olarak
imagenet_net = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1).eval()
filters = imagenet_net.features[0][0].weight.detach()      # (32, 3, 3, 3)
print(tuple(filters.shape))
fig, axes = plt.subplots(4, 8, figsize=(6.4, 3.4))
for k, ax in enumerate(axes.flat):
    f = filters[k].permute(1, 2, 0).numpy()
    ax.imshow((f - f.min()) / (f.max() - f.min() + 1e-9))
    ax.axis("off")
fig.suptitle("ImageNet'te öğrenilmiş EfficientNet-B0'ın ilk katmanı", fontsize=9)
plt.show()

**Python notu.** `.detach()` sayıları autograd'ın dışına alır; böylece gösterilebilirler. Bir filtrenin boyutu (kanal, satır, sütun) iken `imshow` (satır, sütun, kanal) bekler; `.permute(1, 2, 0)` boyutların sırasını değiştirir. Her filtre gösterilmeden önce 0 ile 1 aralığına yayılır. Filtrelerin çoğu küçük kenarlara ya da renk karşıtlıklarına benzer. Bir ağın fotoğraflardan öğrendiği ilk şeyler bunlardır; bu filtreler bir ventrikülün sınırını bulmak için de aynı ölçüde işe yarar.

In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_SD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def as_photos(x, size=128):
    """ED, ES ve ortalamaları üç renk kanalı olarak; ImageNet'teki gibi boyutlandırılmış ve normalleştirilmiş."""
    x = torch.as_tensor(x, dtype=torch.float32)
    x3 = torch.stack([x[:, 0], x[:, 1], (x[:, 0] + x[:, 1]) / 2], dim=1)
    x3 = F.interpolate(x3, size=(size, size), mode="bilinear", align_corners=False)
    return (x3 - IMAGENET_MEAN) / IMAGENET_SD

P = acdc.pairs(GROUPS)
X, y, patients, sets, classes = P["X"], P["y"], P["patient"], P["set"], P["classes"]
photos = as_photos(X)
print(tuple(X.shape), "→", tuple(photos.shape), classes)

**Python notu.** `acdc.pairs` 3. haftanın aracıdır ve 90 hastanın ortadaki üç kesitini, her kesit için 64 × 64 piksellik iki kareyle birlikte döndürür. Her kesitin hangi kümede olduğu da verilir (45 eğitim, 15 doğrulama ve 30 test hastası). `torch.stack(..., dim=1)` üç görüntüyü kanal olarak yan yana koyar. `.view(1, 3, 1, 1)` üç ortalamayı, her biri her pikselde kendi kanalından çıkarılacak biçimde düzenler.

In [ ]:
# 134 numaralı hastanın bir kesiti, ImageNet'in gördüğü biçimiyle
k = int(np.where(patients == "patient134")[0][1])
rgb = photos[k] * IMAGENET_SD[0] + IMAGENET_MEAN[0]
fig, axes = plt.subplots(1, 4, figsize=(8.4, 2.4))
for ax, image, title in zip(axes, [rgb[0], rgb[1], rgb[2], rgb.permute(1, 2, 0)],
                            ["kırmızı: ED", "yeşil: ES", "mavi: ortalama", "birlikte"]):
    ax.imshow(image.clamp(0, 1).numpy(), cmap="gray" if image.ndim == 2 else None)
    ax.set_title(title, fontsize=8)
    ax.axis("off")
fig.suptitle("ACDC, CC BY-NC-SA 4.0", fontsize=7, y=0.04)
plt.show()

**Gözlemle ve düşün.** Sağdaki görüntüde hareket etmeyen yapılar gri görünür, çünkü üç kanal orada aynıdır; diyastol sonu ve sistol sonu karelerinin farklılaştığı yerlerde ise renk ortaya çıkar. Ejeksiyon fraksiyonu çok düşük, dilate kardiyomiyopatili bir hastada kesitin neresinde renk görmeyi beklerdiniz?

**Alıştırma A.** EfficientNet-B0, ImageNet üzerinde eğitildiği hâliyle, yani 1.000 sınıflık son katmanıyla, kaç ağırlığa sahiptir?

In [ ]:
cevap = None   # cevabınız
teach.check("Alıştırma A", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = sum(p.numel() for p in imagenet_net.parameters())
teach.check("Alıştırma A", cevap)

## 2. Öznitelik çıkarımı: Donmuş bir ağ ve basit bir model

*Ders: [Transfer öğrenme: Bir ağın fotoğraflardan öğrendikleri](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-04/ders.html#transfer-learning-what-a-network-learned-from-photographs)*

Öznitelik çıkarımında ImageNet üzerinde eğitilmiş ağ yeniden eğitilmez. Her kesit ağın donmuş katmanlarından bir kez geçer ve son katmandan önceki ortalama havuzlama, kesiti 1.280 sayıyla, yani özniteliklerle tarif eder. Basit bir model, burada bir lojistik regresyon, üç grubu bu sayılardan öğrenir. Bir hasta için karar, 3. haftada olduğu gibi, hastanın üç kesiti üzerinden ortalama olasılığı en yüksek olan gruptur.

**Formüllerden koda.** Bir kesitin öznitelikleri $\mathbf{f} = \text{pool}(\text{features}(x)) \in \mathbb{R}^{1280}$ ile verilir: `net.avgpool(net.features(x)).flatten(1)`. 2. haftanın üç sınıfa genişletilmiş lojistik regresyonu, $z_k = \mathbf{w}_k \cdot \mathbf{f} + b_k$ olmak üzere $p_k = e^{z_k}/\sum_j e^{z_j}$ olasılıklarını verir: `make_pipeline(StandardScaler(), LogisticRegression(C=0.1))`. Burada küçük bir `C` değeri, 1.280 öznitelik ve az sayıda hasta olduğunda ağırlıkları küçük tutar.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 2. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB04_transfer_ogrenme.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-04/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(2)

In [ ]:
def by_patient(prob, mask):
    """Her hastanın kesitleri üzerinden ortalama olasılıklar ve doğru kararların oranı."""
    names = sorted(set(patients[mask]))
    mean_prob = np.array([prob[patients[mask] == n].mean(axis=0) for n in names])
    truth = np.array([y[mask][patients[mask] == n][0] for n in names])
    return float((mean_prob.argmax(axis=1) == truth).mean())

with torch.no_grad():
    features = torch.cat([imagenet_net.avgpool(imagenet_net.features(photos[i:i + 32])).flatten(1)
                          for i in range(0, len(photos), 32)]).numpy()
tr, va, te = sets == "train", sets == "val", sets == "test"
feature_model = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=5000))
feature_model.fit(features[tr], y[tr])
acc_features = round(by_patient(feature_model.predict_proba(features[te]), te), 3)
print(features.shape, "test doğruluğu:", acc_features)

**Python notu.** `with torch.no_grad():` autograd'ın kayıt tutmasını kapatır; çünkü burada hiçbir ağırlık eğitilmez. Bu, bellekten ve zamandan tasarruf sağlar. Kesitler ağdan 32'şerlik gruplar hâlinde geçer (`photos[i:i + 32]`), `torch.cat` ise 9 grubu yeniden birleştirir. `by_patient`, 3. haftadaki gibi her hastanın üç kesitinin olasılıklarının ortalamasını alır ve doğru kararların oranını verir. 270 kesitin özniteliklerini hesaplamak birkaç saniye sürer.

In [ ]:
# 30 test hastasının kararları
names = sorted(set(patients[te]))
prob_te = feature_model.predict_proba(features[te])
decision = [int(prob_te[patients[te] == n].mean(axis=0).argmax()) for n in names]
truth = [int(y[patients == n][0]) for n in names]
print(pd.DataFrame(confusion_matrix(truth, decision), index=classes, columns=classes))

**Gözlemle ve düşün.** Lojistik regresyon hangi grupları birbirine en çok karıştırıyor? Öznitelikler hayvanların, taşıtların ve nesnelerin fotoğraflarından öğrenilmiştir. Yine de dilate bir ventrikülü kalın duvarlı bir ventrikülden neden ayırabilirler?

**Alıştırma B.** Aynı lojistik regresyonun 45 eğitim hastasındaki doğruluğu, her hasta için tek bir kararla hesaplandığında nedir? Sonucu üç ondalık basamağa yuvarlayın ve test doğruluğuyla karşılaştırın.

In [ ]:
veri = alistirma_tablosu("B")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma B", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = round(by_patient(feature_model.predict_proba(features[tr]), tr), 3)
teach.check("Alıştırma B", cevap)

## 3. İnce ayar: Bütün katmanlar yeniden eğitilir

*Ders: [Transfer öğrenme: Bir ağın fotoğraflardan öğrendikleri](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-04/ders.html#transfer-learning-what-a-network-learned-from-photographs)*

İnce ayarda EfficientNet-B0'ın her ağırlığı ImageNet'ten başlar ve küçük bir öğrenme hızıyla kesitlerden öğrenmeyi sürdürür. Dropout içeren ve üç çıktısı olan yeni son katman ise rastgele ağırlıklarla başlar. Eğitim döngüsü 3. haftadakiyle aynıdır: Mini-batch'ler kullanılır, veri çoğaltma için kesitler çeyrek tur döndürülür, her epoch'tan sonra doğrulama kaybı ölçülür ve en iyi epoch'un ağırlıkları saklanır.

**Formüllerden koda.** Her güncellemede dört milyon ağırlığın hepsi gradyanlarının tersi yönünde hareket eder; bunun en basit biçimi $w \leftarrow w - \eta\,\partial L/\partial w$'dir. Adam yöntemi her adımı gradyanların hareketli ortalamalarıyla ölçekler; burada öğrenme hızı $\eta = 0{,}001$'dir: `torch.optim.Adam(net.parameters(), lr=1e-3)`. `net.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(1280, 3))`, 1.000 sınıflık son katmanın yerini alır. `fine_tune`, her epoch'un eğitim ve doğrulama kaybını döndürür.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 3. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB04_transfer_ogrenme.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-04/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(3)

In [ ]:
def fine_tune(net, x_tr, y_tr, x_va, y_va, epochs=15, lr=1e-3, batch=16, seed=RANDOM_STATE):
    """net'in bütün ağırlıklarını eğit; doğrulama kaybının en düşük olduğu epoch'u sakla."""
    optimizer = torch.optim.Adam(net.parameters(), lr=lr, weight_decay=1e-4)
    order_gen = torch.Generator().manual_seed(seed)
    best, best_state, history = float("inf"), None, []
    for epoch in range(epochs):
        net.train()
        order = torch.randperm(len(y_tr), generator=order_gen)
        total = 0.0
        for start in range(0, len(y_tr), batch):
            j = order[start:start + batch]
            turns = int(torch.randint(0, 4, (1,), generator=order_gen))
            loss = F.cross_entropy(net(torch.rot90(x_tr[j], turns, dims=(2, 3))), y_tr[j])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * len(j)
        net.eval()
        with torch.no_grad():
            val_loss = F.cross_entropy(net(x_va), y_va).item()
        history.append((total / len(y_tr), val_loss))
        if val_loss < best:
            best, best_state = val_loss, copy.deepcopy(net.state_dict())
    net.load_state_dict(best_state)
    net.eval()
    return history

y_t = torch.tensor(y, dtype=torch.long)
torch.manual_seed(RANDOM_STATE)
tuned = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
tuned.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(1280, 3))
history = fine_tune(tuned, photos[tr], y_t[tr], photos[va], y_t[va])
with torch.no_grad():
    prob_tuned = F.softmax(tuned(photos[te]), dim=1).numpy()
acc_finetune = round(by_patient(prob_tuned, te), 3)
print("İnce ayarlı ağ, test doğruluğu:", acc_finetune, "· öznitelikler:", acc_features)

**Python notu.** `fine_tune` fonksiyonu 3. haftanın eğitim döngüsünü tek bir fonksiyonda toplar; böylece sonraki bölümler onu başka hastalarla yeniden çağırabilir. `torch.randint(0, 4, (1,), generator=order_gen)` çeyrek tur sayısını, kesitlerin sırasını belirleyen rastgele üreteçten çeker; bu yüzden aynı seed değeri aynı eğitimi verir. `copy.deepcopy(net.state_dict())` en iyi ağırlıkların tam bir kopyasını saklar. 135 eğitim kesitiyle bu hücre bir dizüstü bilgisayarda yaklaşık bir dakika sürer.

In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 3.2))
ax.plot(range(1, len(history) + 1), [h[0] for h in history], color="#1F5F8B", label="eğitim")
ax.plot(range(1, len(history) + 1), [h[1] for h in history], color="#9B1B1B", label="doğrulama")
ax.set_xlabel("epoch")
ax.set_ylabel("çapraz entropi kaybı")
ax.legend()
plt.show()

**Gözlemle ve düşün.** Dersin çalıştırmasında aynı 30 test hastasında ince ayar 0,967, yalnızca öznitelikler ise 0,867 doğruluğa ulaşmıştır. Tek bir hastanın doğru ya da yanlış sınıflanması doğruluğu 0,033 değiştirir. Bir yöntemin diğerinden daha iyi olduğunu söylemeden önce neye ihtiyaç duyardınız?

**İleri düzey (isteğe bağlı).** Aynı ağ rastgele ağırlıklarla da başlayabilir: `efficientnet_b0(weights=None)`. Aşağıdaki hücre bu ağı ince ayarlı ağla tamamen aynı biçimde eğitir; bu da yaklaşık bir dakika daha sürer. Dersin çalıştırmasında bu ağ, grup başına 15 hastayla 0,833 doğruluğa ulaşmış, grup başına beş hastayla ise bir grubun payı olan 0,333'te kalmıştır.

In [ ]:
torch.manual_seed(RANDOM_STATE)
from_random = efficientnet_b0(weights=None)
from_random.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(1280, 3))
fine_tune(from_random, photos[tr], y_t[tr], photos[va], y_t[va])
with torch.no_grad():
    acc_random = round(by_patient(F.softmax(from_random(photos[te]), dim=1).numpy(), te), 3)
print("Rastgele ağırlıklarla, test doğruluğu:", acc_random)

**Alıştırma C.** İnce ayar, 135 eğitim kesiti üzerinde 16'şar kesitlik mini-batch'lerle 15 epoch çalışır; bir epoch'un son mini-batch'i daha küçüktür. Ağırlıklar toplam kaç kez güncellenir?

In [ ]:
cevap = None   # cevabınız
teach.check("Alıştırma C", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = math.ceil(135 / 16) * 15
teach.check("Alıştırma C", cevap)

## 4. Az hastayla adil bir karşılaştırma

*Ders: [Az sayıda etiketli hastayla öğrenmek](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-04/ders.html#learning-from-few-labelled-patients)*

Az sayıda hastadan elde edilen bir sonuç, hangi hastaların seçildiğine bağlıdır. Bu yüzden adil bir karşılaştırmada her gruptan k eğitim hastası birkaç kez çekilir ve her yöntem aynı çekilişlerle, aynı doğrulama ve test hastalarıyla çalıştırılır; sonuçların ortalaması da dağılımıyla birlikte bildirilir. Bu bölümde dersin çekilişleri öznitelik yöntemi için tekrarlanır; bu işlem birkaç saniye sürer. Ardından sonuçlar, ders sayfasındaki diğer üç yöntemin sonuçlarıyla aynı grafikte gösterilir; bu üç yöntemin eğitimi derste yaklaşık on yedi dakika sürmüştür.

**Formüllerden koda.** Grup başına k hasta ve $d$ çekilişi için doğruluk $a_{k,d}$ ile gösterilir; eğri, en küçük ve en büyük değeriyle birlikte $\bar{a}_k = \frac{1}{D}\sum_d a_{k,d}$ değerini gösterir: `np.mean(curve[k])`, `min(curve[k])` ve `max(curve[k])`. Çekilişler `np.random.default_rng(RANDOM_STATE)` ile sabit bir sırayla yapılır: k = 1, 2, 5, 10 ve 15; 15'ten küçük her k için beş çekiliş.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 4. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB04_transfer_ogrenme.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-04/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(4)

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
train_names = sorted(set(patients[tr]))
group_of = {n: int(y[patients == n][0]) for n in train_names}
curve = {}
for k in (1, 2, 5, 10, 15):
    curve[k] = []
    for draw in range(5 if k < 15 else 1):
        chosen = []
        for g in range(3):
            pool = [n for n in train_names if group_of[n] == g]
            chosen += sorted(rng.choice(pool, size=k, replace=False).tolist())
        rows = np.isin(patients, chosen)
        small = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=5000))
        small.fit(features[rows], y[rows])
        curve[k].append(by_patient(small.predict_proba(features[te]), te))
for k, accs in curve.items():
    print(k, "grup başına:", np.round(accs, 3), "ortalama", round(float(np.mean(accs)), 3))

**Python notu.** `rng.choice(pool, size=k, replace=False)` bir listeden k farklı ad çeker; `replace=False` aynı hastanın iki kez çekilmesini önler. `np.isin(patients, chosen)` seçilen hastaların kesitlerini işaretler. `range(5 if k < 15 else 1)` beş kez tekrarlar; k = 15 bunun dışındadır: ACDC'nin eğitim kümesinde her gruptan tam 15 hasta bulunduğu için yalnızca tek bir çekiliş mümkündür.

In [ ]:
# Diğer üç yöntem: Aynı çekilişlerdeki ortalama doğruluk, dersin çalıştırmasından
COURSE_RUN = {"sıfırdan eğitilen 3. hafta ağı": [0.480, 0.573, 0.720, 0.820, 0.933],
              "rastgele ağırlıklarla EfficientNet-B0": [0.333, 0.333, 0.333, 0.740, 0.833],
              "ince ayarlı EfficientNet-B0": [0.480, 0.607, 0.793, 0.860, 0.967]}
ks = list(curve)
fig, ax = plt.subplots(figsize=(6.2, 3.6))
ax.fill_between(ks, [min(curve[k]) for k in ks], [max(curve[k]) for k in ks], color="#B97813", alpha=0.2)
ax.plot(ks, [np.mean(curve[k]) for k in ks], "o-", color="#B97813", label="ImageNet öznitelikleri (bu defter)")
for (name, accs), color in zip(COURSE_RUN.items(), ["#566A73", "#888888", "#1F5F8B"]):
    ax.plot(ks, accs, "s--", color=color, label=name, lw=1)
ax.set_xlabel("grup başına eğitim hastası")
ax.set_ylabel("doğruluk, 30 test hastası")
ax.set_ylim(0.2, 1.02)
ax.legend(fontsize=7)
plt.show()

**Gözlemle ve düşün.** Grup başına bir ya da iki hastayla özniteliklerin bandı geniştir: Aynı yöntem farklı hastalarla çok farklı sonuçlar verir. Grup başına on incelemeyi etiketlemiş yeni bir laboratuvar için hangi yöntemi seçerdiniz ve ortalama doğruluğun yanında neyi bildirirdiniz?

**Alıştırma D.** Grup başına beş hastayla yapılan beş çekilişte öznitelik yönteminin ortalama doğruluğu nedir? Sonucu üç ondalık basamağa yuvarlayın.

In [ ]:
veri = alistirma_tablosu("D")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma D", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = round(float(np.mean(curve[5])), 3)
teach.check("Alıştırma D", cevap)

## 5. CAMUS: Dizi olarak kareler ve sınırlar

*Ders: [Segmentasyon: Her piksel için bir etiket](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-04/ders.html#segmentation-a-label-for-every-pixel)*

CAMUS, Saint-Étienne Üniversite Hastanesindeki 500 hastanın apikal iki boşluk ve dört boşluk görüntülerini içerir. Her görüntü için diyastol sonu ve sistol sonu kareleri uzman sınırlarıyla birlikte gelir. Eğitim 300 hastayı kullanmaktadır: Yazarların eğitim kümesinden 200 hasta, doğrulama kümesindeki 50 hasta ve test kümesindeki 50 hasta. Ağ için her kare 96 × 96 piksele küçültülür; sınırlar ise en yakın etiketle küçültülür, böylece iki sınıfın arasında yeni etiketler ortaya çıkmaz.

**Formüllerden koda.** Bir kare $x \in [0, 1]^{96 \times 96}$ dizisi, sınırı ise $m \in \{0, 1, 2, 3\}^{96 \times 96}$ etiket dizisidir: `images[i]` ve `masks[i]`. Ağın hedefi sol ventrikül kavitesi, yani $t = [m = 1]$'dir: `masks == 1`. `camus.frames`, her hasta için dört olmak üzere 1.200 kareyi hastası, görüntüsü, fazı ve kümesiyle birlikte verir.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 5. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB04_transfer_ogrenme.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-04/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(5)

In [ ]:
frames_ = camus.frames(camus.patients(), size=96)
images, masks = frames_["images"], frames_["masks"]
frame_patient, frame_view, frame_phase, frame_split = frames_["patient"], frames_["view"], frames_["phase"], frames_["split"]
frame_shape = frames_["shape"]
camus_table = camus.table()
print(images.shape, masks.shape, {s: int((frame_split == s).sum()) for s in ("training", "validation", "testing")})
print(camus_table.set_index("patient").loc["patient0201"])

**Python notu.** `camus.frames`, 0. bölümde indirilen dosyalardan her hastanın dört karesini okur ve küçültür; 1.200 kareyi okumak yaklaşık on beş saniye sürer. `camus.table` Info dosyalarını okur: Cinsiyet, yaş, iki görüntünün kalitesi ve yazarların uzman sınırlarından hesapladığı ejeksiyon fraksiyonu (EF). `.loc["patient0201"]` haftanın vakasının satırını alır.

In [ ]:
# 201 numaralı hastanın dört karesi özgün boyutlarında, uzman sınırlarıyla
fig, axes = plt.subplots(1, 4, figsize=(9.6, 3.2))
for ax, (view, phase) in zip(axes, [("2CH", "ED"), ("2CH", "ES"), ("4CH", "ED"), ("4CH", "ES")]):
    im, lab, _ = camus.load("patient0201", view, phase)
    ax.imshow(im, cmap="gray")
    for label, color in ((1, "#E5797B"), (2, "#F9AB00"), (3, "#7DB4DD")):
        ax.contour(lab == label, levels=[0.5], colors=color, linewidths=1)
    ax.set_title(f"{view} {phase}", fontsize=9)
    ax.axis("off")
fig.suptitle("kırmızı: SV kavitesi · sarı: miyokard · mavi: sol atriyum · CAMUS, CC BY-NC-SA 4.0", fontsize=8, y=0.06)
plt.show()

**Gözlemle ve düşün.** Sistol sonundaki kavite daha küçüktür ve sınırı miyokarda daha yakındır. İki görüntü ventrikülü birbirine yaklaşık 60 derece açıyla duran iki düzlemde keser. Disk yöntemi neden tek bir görüntüye değil, iki görüntüye ihtiyaç duyar?

**Alıştırma E.** 96 × 96 piksellik 800 eğitim karesinin bütün piksellerinin yüzde kaçı sol ventrikül kavitesine aittir? Sonucu bir ondalık basamağa yuvarlayın.

In [ ]:
veri = alistirma_tablosu("E")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma E", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = round(100 * float((masks[frame_split == "training"] == 1).mean()), 1)
teach.check("Alıştırma E", cevap)

## 6. Dice katsayısı

*Ders: [Segmentasyon: Her piksel için bir etiket](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-04/ders.html#segmentation-a-label-for-every-pixel)*

Kavite piksellerin onda birinden azını kaplar; bu yüzden her piksele arka plan diyen bir ağ, piksellerin yüzde 90'ından fazlasında haklı olur ama yine de işe yaramaz. Dice katsayısı iki sınırı doğrudan karşılaştırır: İki sınırın örtüşmesinin iki katı, boyutlarının toplamına bölünür. Katsayı aynı sınırlar için 1, hiç ortak pikseli olmayan sınırlar için 0'dır.

**Formüllerden koda.** $D = 2|A \cap B|/(|A| + |B|)$ ile hesaplanır: `2 * (a & b).sum() / (a.sum() + b.sum())`. `a & b`, iki maskenin de doğru olduğu yerlerde doğrudur. `np.roll(a, 3, axis=1)` bir maskeyi, sınır 3 piksel fazla çizilmiş gibi 3 sütun sağa kaydırır.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 6. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB04_transfer_ogrenme.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-04/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(6)

In [ ]:
def dice(pred, true):
    """Her karenin Dice katsayısı: aynı boyutta iki mantıksal dizi."""
    inter = (pred & true).sum(axis=(-2, -1))
    return 2 * inter / np.maximum(pred.sum(axis=(-2, -1)) + true.sum(axis=(-2, -1)), 1)


def frame_of(patient, view, phase):
    return int(np.where((frame_patient == patient) & (frame_view == view) & (frame_phase == phase))[0][0])


rows = []
for phase in ("ED", "ES"):
    a = masks[frame_of("patient0201", "4CH", phase)] == 1
    for shift in range(7):
        rows.append(dict(phase=phase, shift=shift, dice=round(float(dice(np.roll(a, shift, axis=1), a)), 3)))
shifts = pd.DataFrame(rows).pivot(index="shift", columns="phase", values="dice")
print(shifts)

**Python notu.** `axis=(-2, -1)` toplamı son iki boyut, yani satırlar ve sütunlar üzerinden alır; böylece aynı fonksiyon tek bir kare için bir Dice katsayısı, birden çok kare için ise her kareye bir Dice katsayısı verir. `np.maximum(..., 1)`, iki maske de boş olduğunda sıfıra bölmeyi önler. `.pivot`, 14 satırlık uzun tabloyu her kayma için bir satır ve her faz için bir sütun içeren bir tabloya dönüştürür.

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 3.2))
for phase, color in (("ED", "#1F5F8B"), ("ES", "#9B1B1B")):
    ax.plot(shifts.index, shifts[phase], "o-", color=color, label=phase)
ax.set_xlabel("sınırın kayması (96 pikselde)")
ax.set_ylabel("Dice")
ax.legend()
plt.show()

**Gözlemle ve düşün.** Aynı kayma, sistol sonunda Dice katsayısını diyastol sonundakinden daha çok düşürür. Ağlar çoğu zaman bütün karelerin ortalama Dice katsayısıyla değerlendirilir. Bir laboratuvar neden yalnızca sistol sonu karelerinin Dice katsayısını da ayrıca sormalıdır?

**Alıştırma F.** 201 numaralı hastanın dört boşluk görüntüsünde, ikisi de özgün boyutunda olmak üzere, diyastol sonu ve sistol sonu kaviteleri arasındaki Dice katsayısı nedir? Sonucu üç ondalık basamağa yuvarlayın.

In [ ]:
veri = alistirma_tablosu("F")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma F", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
_, lab_ed, _ = camus.load("patient0201", "4CH", "ED")
_, lab_es, _ = camus.load("patient0201", "4CH", "ES")
cevap = round(float(dice(lab_ed == 1, lab_es == 1)), 3)
teach.check("Alıştırma F", cevap)

## 7. Sol ventrikül kavitesi için bir U-Net

*Ders: [CAMUS ekokardiyogramlarında bir U-Net](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-04/ders.html#a-u-net-on-the-echocardiograms-of-camus)*

Bir U-Net, adındaki U harfinin biçimindedir. Kodlayıcısı kareyi üç adımda 96 pikselden 12 piksele küçültür ve kanal sayısını 8'den 64'e çıkarır; böylece en alttaki her piksel kalbin büyük bir bölümünü görür. Kod çözücüsü kareyi üç adımda yeniden büyütür ve her düzeyde bir atlama bağlantısı, kodlayıcının ince ayrıntısını karşıya aktarır. Son katman her piksel için iki sayı verir: Arka plan ve kavite. Kayıp, her pikselin çapraz entropisini ve yumuşak Dice kaybını toplar; yumuşak Dice kaybı küçük kaviteye büyük arka plan kadar ağırlık verir.

**Formüllerden koda.** Yumuşak Dice kaybı, $j$ pikselinin kavite olasılığı $p_j$ ve hedefi $t_j$ olmak üzere $L_D = 1 - \frac{2\sum_j p_j t_j + 1}{\sum_j p_j + \sum_j t_j + 1}$ ile hesaplanır: `soft_dice_loss`. Bir mini-batch'in kaybı $L = L_{CE} + L_D$'dir: `F.cross_entropy(out, target) + soft_dice_loss(out, target)`. `torch.cat([self.up3(b), d3], dim=1)` bir atlama bağlantısıdır: Büyütülmüş harita ile kodlayıcının haritası kanal olarak üst üste konur. `OneCycleLR`, öğrenme hızını 15 epoch boyunca önce 0,003'e çıkarır, sonra yeniden düşürür.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 7. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB04_transfer_ogrenme.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-04/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(7)

In [ ]:
def unet_block(c_in, c_out):
    return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
                         nn.Conv2d(c_out, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU())


class UNet(nn.Module):
    """Üç düzey aşağı, bir dar boğaz ve atlama bağlantılarıyla üç düzey yukarı."""

    def __init__(self, c=8, n_classes=2):
        super().__init__()
        self.down1, self.down2, self.down3 = unet_block(1, c), unet_block(c, 2 * c), unet_block(2 * c, 4 * c)
        self.bottom = unet_block(4 * c, 8 * c)
        self.up3, self.dec3 = nn.ConvTranspose2d(8 * c, 4 * c, 2, stride=2), unet_block(8 * c, 4 * c)
        self.up2, self.dec2 = nn.ConvTranspose2d(4 * c, 2 * c, 2, stride=2), unet_block(4 * c, 2 * c)
        self.up1, self.dec1 = nn.ConvTranspose2d(2 * c, c, 2, stride=2), unet_block(2 * c, c)
        self.out = nn.Conv2d(c, n_classes, 1)

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(F.max_pool2d(d1, 2))
        d3 = self.down3(F.max_pool2d(d2, 2))
        b = self.bottom(F.max_pool2d(d3, 2))
        u3 = self.dec3(torch.cat([self.up3(b), d3], dim=1))
        u2 = self.dec2(torch.cat([self.up2(u3), d2], dim=1))
        u1 = self.dec1(torch.cat([self.up1(u2), d1], dim=1))
        return self.out(u1)


def soft_dice_loss(logits, target):
    p = F.softmax(logits, dim=1)[:, 1]
    t = target.float()
    inter = (p * t).sum(dim=(1, 2))
    return 1 - ((2 * inter + 1) / (p.sum(dim=(1, 2)) + t.sum(dim=(1, 2)) + 1)).mean()


def dice(pred, true):
    """Her karenin Dice katsayısı: aynı boyutta iki mantıksal dizi."""
    inter = (pred & true).sum(axis=(-2, -1))
    return 2 * inter / np.maximum(pred.sum(axis=(-2, -1)) + true.sum(axis=(-2, -1)), 1)

print("U-Net'in eğitilebilir parametreleri:", sum(p.numel() for p in UNet().parameters()))

**Python notu.** `class UNet(nn.Module)`, 3. haftanın ağıyla aynı kalıbı izler: `__init__` katmanları oluşturur, `forward` ise bir karenin bu katmanlardan nasıl geçeceğini söyler. `unet_block` fonksiyonu bir düzeyin iki evrişimini kurar; böylece sınıf onu yedi kez yeniden kullanabilir. `nn.ConvTranspose2d(..., 2, stride=2)` bir haritanın boyutunu iki katına çıkarır. `d1`, `d2` ve `d3`, kodlayıcının haritalarını kod çözücü onlara ihtiyaç duyana kadar saklar.

In [ ]:
cavity = masks == 1
tr, va, te = frame_split == "training", frame_split == "validation", frame_split == "testing"
X_img = torch.tensor(images)[:, None]
Y_cav = torch.tensor(cavity.astype(np.int64))
torch.manual_seed(RANDOM_STATE)
unet = UNet()
epochs, batch = 15, 16
optimizer = torch.optim.Adam(unet.parameters(), lr=3e-3)
steps = epochs * ((int(tr.sum()) + batch - 1) // batch)
schedule = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=3e-3, total_steps=steps)
order_gen = torch.Generator().manual_seed(RANDOM_STATE)
X_tr, Y_tr = X_img[tr], Y_cav[tr]
best, best_state = -1.0, None
for epoch in range(epochs):
    unet.train()
    order = torch.randperm(len(X_tr), generator=order_gen)
    for start in range(0, len(X_tr), batch):
        j = order[start:start + batch]
        out = unet(X_tr[j])
        loss = F.cross_entropy(out, Y_tr[j]) + soft_dice_loss(out, Y_tr[j])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        schedule.step()
    unet.eval()
    with torch.no_grad():
        val_dice = float(dice(unet(X_img[va]).argmax(dim=1).numpy() == 1, cavity[va]).mean())
    if val_dice > best:
        best, best_state, best_epoch = val_dice, copy.deepcopy(unet.state_dict()), epoch + 1
unet.load_state_dict(best_state)
unet.eval()
with torch.no_grad():
    test_prob = F.softmax(unet(X_img[te]), dim=1)[:, 1].numpy()

test_dice = round(float(dice(test_prob > 0.5, cavity[te]).mean()), 3)
print("En iyi epoch:", best_epoch, "· 200 test karesinin Dice katsayısı:", test_dice)

**Python notu.** Eğitim döngüsü 3. haftadakiyle aynıdır; yalnızca iki farkı vardır. `OneCycleLR` zamanlayıcısı öğrenme hızını her mini-batch'ten sonra değiştirir ve en iyi epoch, doğrulama karelerinin kaybıyla değil, Dice katsayısıyla seçilir. `unet(...).argmax(dim=1)` her pikselde çıktısı daha büyük olan sınıfı verir. Eğitim bir dizüstü bilgisayarda yaklaşık üç dakika sürer; dersin çalıştırmasında en iyi epoch 13 olmuş ve test kareleri 0,924 ortalama Dice katsayısına ulaşmıştır.

In [ ]:
per_frame = dice(test_prob > 0.5, cavity[te])
quality = {}
for _, r in camus_table.iterrows():
    quality[(r["patient"], "2CH")], quality[(r["patient"], "4CH")] = r["quality_2ch"], r["quality_4ch"]
test_frames = pd.DataFrame({"patient": frame_patient[te], "view": frame_view[te], "phase": frame_phase[te], "dice": per_frame})
test_frames["quality"] = [quality[(p, v)] for p, v in zip(test_frames["patient"], test_frames["view"])]
print(test_frames.groupby(["view", "phase"])["dice"].mean().round(3).unstack())
print(test_frames.groupby("quality")["dice"].agg(["count", "mean"]).round(3))

**Gözlemle ve düşün.** Sistol sonundaki iki boşluk görüntüsü en zayıf gruptur ve kalitesi kötü kareler iyi karelerden daha kötü çizilmektedir. Ağı kullanmak isteyen bir laboratuvara bu iki bulgudan hangisini, hangi sözcüklerle bildirirdiniz?

In [ ]:
# Haftanın vakası: Uzman (kırmızı) ve ağ (mavi), özgün boyutta
test_index = np.where(te)[0]
net_masks = {}
for k, i in enumerate(test_index):
    net_masks.setdefault(frame_patient[i], {})[(frame_view[i], frame_phase[i])] = camus.to_original(test_prob[k], frame_shape[i])
fig, axes = plt.subplots(1, 4, figsize=(9.6, 3.2))
for ax, key in zip(axes, [("2CH", "ED"), ("2CH", "ES"), ("4CH", "ED"), ("4CH", "ES")]):
    im, lab, _ = camus.load("patient0201", *key)
    ax.imshow(im, cmap="gray")
    ax.contour(lab == 1, levels=[0.5], colors="#E5797B", linewidths=1.2)
    ax.contour(net_masks["patient0201"][key], levels=[0.5], colors="#7DB4DD", linewidths=1.2)
    ax.set_title(f"{key[0]} {key[1]}: Dice {dice(net_masks['patient0201'][key].astype(bool), lab == 1):.3f}", fontsize=8)
    ax.axis("off")
fig.suptitle("CAMUS, CC BY-NC-SA 4.0", fontsize=7, y=0.06)
plt.show()

**Python notu.** `camus.to_original`, ağın etiketlerini değil olasılıklarını yaklaşık 520 × 630 piksellik özgün kareye büyütür, 0,5'te eşikler ve delikleri doldurulmuş en büyük bağlantılı bölgeyi tutar. Olasılıkları büyütmek düzgün bir sınır verir; etiketleri büyütmek ise beşer piksellik basamaklar oluştururdu. `net_masks` sözlüklerden oluşan bir sözlüktür: Her hasta için bir girdi ve bu girdinin içinde her görüntü ve faz çifti için bir maske vardır.

**Alıştırma G.** İlk düzeyinde 8 yerine 16 kanal bulunan bir U-Net'in, yani `UNet(c=16)` ağının kaç eğitilebilir parametresi vardır?

In [ ]:
cevap = None   # cevabınız
teach.check("Alıştırma G", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = sum(p.numel() for p in UNet(c=16).parameters() if p.requires_grad)
teach.check("Alıştırma G", cevap)

## 8. Disk yöntemiyle hacimler ve ejeksiyon fraksiyonu

*Ders: [Disk yöntemiyle ejeksiyon fraksiyonu](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-04/ders.html#the-ejection-fraction-by-the-method-of-discs)*

Kılavuzlardaki modifiye Simpson yöntemi olan biplan disk yöntemi, ventrikülü uzun ekseni boyunca, mitral halkadan apekse kadar 20 eliptik diske böler. İki boşluk görüntüsü her diskin bir çapını, dört boşluk görüntüsü ise diğer çapını verir. Bir diskin kalınlığı uzun eksenin yirmide biridir; CAMUS'un referans betiğinde iki hacim için de dört sınırın en uzunundan hesaplanan kalınlık kullanılır.

**Formüllerden koda.** $V = \frac{\pi}{4}\sum_{i=1}^{20} a_i\,b_i\,\frac{L}{20}$ mm³ cinsinden hesaplanır ve mL için 1.000'e bölünür: `np.pi / 4 * np.sum(a * b) * step / 1000`. $EF = 100\,(EDV - ESV)/EDV$: `camus.ejection_fraction(edv, esv)`. `camus.discs(mask, spacing)`, 20 çapı mm cinsinden, bir diskin kalınlığını mm cinsinden, mitral halkanın ortasını ve apeksi döndürür.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 8. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB04_transfer_ogrenme.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-04/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(8)

In [ ]:
outline, discs_of = {}, {}
for view in ("2CH", "4CH"):
    for phase in ("ED", "ES"):
        _, lab, sp = camus.load("patient0201", view, phase)
        outline[(view, phase)] = (lab == 1, sp)
        discs_of[(view, phase)] = camus.discs(lab == 1, sp)
        d, step, base, apex = discs_of[(view, phase)]
        print(view, phase, "L =", round(step * 20, 1), "mm · kalınlık", round(step, 2), "mm · en geniş disk", round(d.max(), 1), "mm")
step = max(x[1] for x in discs_of.values())
edv = np.pi / 4 * np.sum(discs_of[("2CH", "ED")][0] * discs_of[("4CH", "ED")][0]) * step / 1000
esv = np.pi / 4 * np.sum(discs_of[("2CH", "ES")][0] * discs_of[("4CH", "ES")][0]) * step / 1000
print("Elle:", "EDV", round(edv), "mL · ESV", round(esv), "mL · EF", round(camus.ejection_fraction(round(edv), round(esv)), 1), "%")
print("Ders aracı:", camus.measure("patient0201"))

**Python notu.** `camus.discs`, CAMUS'un referans betiğini izler: Mitral halka, sınırın en uzun ve neredeyse düz bölümüdür; apeks, bu bölümün ortasına en uzak noktadır ve 20 çap, ikisini birleştiren çizgiye dik olarak ölçülür. `discs_of[("2CH", "ED")][0]`, döndürülen dört değerin ilkini, yani çapları alır. `max(x[1] for x in discs_of.values())` dört sınırın en büyük kalınlığını alır. Hacimler, referanstaki gibi, EF hesaplanmadan önce tam mL'ye yuvarlanır.

In [ ]:
# Diyastol sonundaki dört boşluk görüntüsünde uzun eksen ve 20 disk
im, lab, sp = camus.load("patient0201", "4CH", "ED")
d, step_4ed, base, apex = discs_of[("4CH", "ED")]
axis = apex - base
fig, ax = plt.subplots(figsize=(4.4, 4.4))
ax.imshow(im, cmap="gray")
ax.contour(lab == 1, levels=[0.5], colors="#E5797B", linewidths=1)
ax.plot([base[1], apex[1]], [base[0], apex[0]], color="#F9AB00", lw=1.5)
normal = np.array([-axis[1], axis[0]]) / np.linalg.norm(axis)
for f, width in zip(np.linspace(0, 1, 20, endpoint=False), d):
    c = base + f * axis
    half = width / 2 / sp[0]
    ax.plot([c[1] - half * normal[1], c[1] + half * normal[1]], [c[0] - half * normal[0], c[0] + half * normal[0]], color="#7DB4DD", lw=0.8)
ax.set_title("4CH ED: uzun eksen ve diskler (eksene ortalanmış)", fontsize=8)
ax.axis("off")
plt.show()

In [ ]:
net_case = camus.measure("patient0201", net_masks["patient0201"])
print(pd.DataFrame([camus.measure("patient0201"), net_case], index=["uzman", "ağ"]))

**Gözlemle ve düşün.** Ağ, hem diyastol sonunda hem de sistol sonunda kaviteyi uzmandan daha büyük çizer; yine de EF'si uzmanınkine yakındır. Hangi klinik karar yalnızca EF'ye değil, hacimlere de bağlıdır?

**Alıştırma H.** 201 numaralı hastayı uzman sınırları üzerinde her faz için ayrı bir kalınlıkla ölçün: `camus.measure("patient0201", thickness="phase")`. Sistol sonu hacmi kaç mL'dir?

In [ ]:
veri = alistirma_tablosu("H")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma H", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = camus.measure("patient0201", thickness="phase")["esv_ml"]
teach.check("Alıştırma H", cevap)

## 9. Uzmanla uyum: Bland-Altman

*Ders: [Uzmanla uyum: Bland-Altman](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-04/ders.html#agreement-with-the-expert-bland-altman)*

İki ölçüm yöntemi birbiriyle yakından ilişkili olduğu hâlde ölçümleri pek çok puan farklı olabilir. Bland ve Altman'ın önerdiği grafikte iki yöntemin farkı, iki ölçümün ortalamasına karşı çizilir. Ortalama fark (bias) sistematik bir hatayı gösterir; ortalama farkın iki yanına farkların standart sapmasının 1,96 katı eklenerek bulunan uyum sınırları ise bir hastanın bir ölçümünün diğerinden ne kadar uzağa düşebileceğini gösterir. Bir laboratuvar hangi sınırları kabul edeceğine testten önce karar verir.

**Formüllerden koda.** 50 test hastası için $d_i = EF_{\text{ağ},i} - EF_{\text{uzman},i}$ olmak üzere $\bar{d}$ `diff.mean()` ile, $s_d$ ise `diff.std(ddof=1)` ile hesaplanır; sınırlar $\bar{d} \pm 1{,}96\,s_d$'dir. Pearson $r$ katsayısı `np.corrcoef(ef_net, ef_expert)[0, 1]` ile bulunur. `ef_range`, `np.select` ile her EF'yi kılavuzun üç aralığından birine yerleştirir; `np.select` sağlanan ilk koşulu seçer. Aralıklar yüzde 40 ve altı, yüzde 40'ın üzeri ile yüzde 50'nin altı ve yüzde 50 ve üzeridir.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 9. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB04_transfer_ogrenme.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-04/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(9)

In [ ]:
rows = []
for p in camus.patients("testing"):
    ref, net = camus.measure(p), camus.measure(p, net_masks[p])
    rows.append(dict(patient=p, ef_expert=ref["ef"], ef_net=net["ef"], edv_expert=ref["edv_ml"], edv_net=net["edv_ml"],
                     esv_expert=ref["esv_ml"], esv_net=net["esv_ml"]))
agreement = pd.DataFrame(rows)
diff = agreement["ef_net"] - agreement["ef_expert"]
bias, sd = float(diff.mean()), float(diff.std(ddof=1))
r = float(np.corrcoef(agreement["ef_net"], agreement["ef_expert"])[0, 1])
print("EF: ortalama fark", round(bias, 2), "· SS", round(sd, 2), "· sınırlar", round(bias - 1.96 * sd, 1), "ile", round(bias + 1.96 * sd, 1), "· r", round(r, 3))

**Python notu.** Her biri dört sınırdan oluşan 100 takımı, yani 50 hastayı hem uzmanın hem de ağın sınırlarıyla ölçmek yaklaşık bir dakika sürer; çünkü disk yöntemi her sınırda mitral halkayı arar. `ddof=1`, bir örneklemde olduğu gibi n − 1'e böler. `agreement` tablosu hacimleri de tutar; böylece hacimlerin uyumu haftalık projede incelenebilir.

In [ ]:
mean_ef = (agreement["ef_net"] + agreement["ef_expert"]) / 2
fig, axes = plt.subplots(1, 2, figsize=(9.4, 3.8))
axes[0].scatter(agreement["ef_expert"], agreement["ef_net"], s=16, color="#1F5F8B")
axes[0].plot([10, 80], [10, 80], color="#888888", lw=1)
axes[0].set_xlabel("EF, uzman (%)")
axes[0].set_ylabel("EF, ağ (%)")
axes[0].set_title(f"r = {r:.2f}", fontsize=9)
axes[1].scatter(mean_ef, diff, s=16, color="#1F5F8B")
for value, style in ((bias, "-"), (bias - 1.96 * sd, "--"), (bias + 1.96 * sd, "--")):
    axes[1].axhline(value, color="#9B1B1B", ls=style, lw=1)
axes[1].set_xlabel("iki EF'nin ortalaması (%)")
axes[1].set_ylabel("ağ eksi uzman (puan)")
axes[1].set_title("Bland-Altman", fontsize=9)
plt.tight_layout()
plt.show()
labels = ["≤40", "41-49", "≥50"]


def ef_range(ef):
    """Kalp yetersizliği kılavuzuna göre her EF'nin aralığı: 40 ve altı, 40'ın üzeri ve 50'nin altı, 50 ve üzeri."""
    return pd.Categorical(np.select([ef <= 40, ef < 50], labels[:2], labels[2]), categories=labels)


print(pd.crosstab(ef_range(agreement["ef_expert"]), ef_range(agreement["ef_net"]),
                  rownames=["uzman"], colnames=["ağ"], dropna=False))

**Gözlemle ve düşün.** Yaklaşık 0,8'lik bir korelasyon iyi görünür; ancak uyum sınırları yaklaşık 30 puanlık bir aralığa yayılır ve hastaların yaklaşık üçte biri EF aralığını değiştirir. Çapraz tablodaki hangi hastalar bir tedavi kararı için en önemlidir ve otomatik EF bir rapora girmeden önce neyi şart koşardınız?

**Alıştırma I.** 50 test hastasını uzman sınırları üzerinde iki kez, `thickness="phase"` ve `thickness="camus"` seçenekleriyle ölçün. EF'nin ortalama farkı (phase eksi camus) kaç puandır? Sonucu iki ondalık basamağa yuvarlayın.

In [ ]:
veri = alistirma_tablosu("I")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma I", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
phase_minus_camus = [camus.measure(p, thickness="phase")["ef"] - camus.measure(p, thickness="camus")["ef"]
                     for p in camus.patients("testing")]
cevap = round(float(np.mean(phase_minus_camus)), 2)
teach.check("Alıştırma I", cevap)

## 10. YZ asistanıyla çalışmak, 4. bölüm: Bir ölçüm hattını denetlemek

*Ders: [YZ asistanıyla çalışmak, 4. bölüm: Bir ölçüm hattını denetlemek](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-04/ders.html#ai-pair-programming-part-4-reviewing-a-measurement-pipeline)*

Bir katılımcı, bir YZ asistanından *201 numaralı hastanın EF'sini sınırlardan ölçmesini ve ağın uzmanla uyumunu bildirmesini* istemiştir. Aşağıdaki kod asistanın cevabıdır. Kod çalışır ve makul görünen bir EF yazdırır. Ona güvenmeden önce kodu ölçüm hattı denetiminin altı sorusuyla okuyun: Kümeler hastaya göre mi ayrılmış? Maskeler nasıl yeniden boyutlandırılmış? Bir pikselin gerçek büyüklüğü kullanılmış mı? Dice katsayısı görüntüye ve faza göre verilmiş mi? Hacimler hangi kuralla hesaplanmış? Uyum mu bildirilmiş, yoksa yalnızca korelasyon mu?

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 10. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB04_transfer_ogrenme.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-04/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(10)

In [ ]:
# Bir YZ asistanının kodu: 96 piksellik kareler üzerinde ölçülen 201 numaralı hastanın EF'si
small = camus.frames(["patient0201"], size=96)
outlines = {(v, ph): small["masks"][i] > 0 for i, (v, ph) in enumerate(zip(small["view"], small["phase"]))}
pixel = (1.0, 1.0)
edv_a, esv_a = camus.volumes(outlines[("2CH", "ED")], outlines[("2CH", "ES")], pixel,
                             outlines[("4CH", "ED")], outlines[("4CH", "ES")], pixel)
print("EF:", round(camus.ejection_fraction(edv_a, esv_a), 1), "%")
# ve 50 test hastası için asistan şunu bildirdi:
print("Uzmanla uyum: r =", round(float(np.corrcoef(agreement["ef_net"], agreement["ef_expert"])[0, 1]), 2))

Kodun üç sorunu ve bir eksiği vardır. `> 0` bütün etiketleri tutar; bu yüzden sınır, kaviteyle birlikte miyokardı ve sol atriyumu da içerir. 96 piksellik maskeler bir piksel 1 mm'ymiş gibi ölçülmüştür; oysa küçültülmüş karede bir piksel yaklaşık 1,7 × 2,0 mm boyutundadır. EF bir oran olduğu için bu hatayı gizler, ancak her hacim yanlıştır. Ayrıca bir korelasyon, uyum olarak bildirilmiştir. Eksik olan şudur: Asistan hacimleri yazdırmamıştır; yazdırsaydı bir şeylerin yanlış olduğu hemen görülürdü.

**Alıştırma J.** Hattı düzeltin: 201 numaralı hastayı, karelerin özgün boyutunda ve bir pikselin gerçek büyüklüğüyle, uzmanın çizdiği kavite (etiket 1) üzerinde ölçün. Diyastol sonu hacmi kaç mL'dir?

In [ ]:
veri = alistirma_tablosu("J")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma J", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = camus.measure("patient0201")["edv_ml"]
teach.check("Alıştırma J", cevap)

Etkileşimli labdaki [prompt atölyesinde](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-04/lab.html#part-f) böyle bir ölçümün promptu, prompt kartının yedi alanıyla yazılır. *Veri* alanı bir pikselin büyüklüğünü ve etiketlerin anlamını belirtir; *Test* alanı ise EF'nin yanında hacimleri de ister.

## Alanlara göre uygulama görevleri

| Alan | Görev |
|---|---|
| Kalp yetersizliği | 9. bölümdeki `agreement` tablosunu kullanın ve EF aralığı (yüzde 40'a kadar, yüzde 41-49, yüzde 50 ya da üzeri) ağ ile uzman arasında farklı olan hastaları sayın. Bu hastaların hangilerinde fark bir tedavi kararını değiştirirdi? |
| Girişimsel kardiyoloji | CAMUS'ta sol atriyumun sınırı da çizilmiştir (etiket 3). 7. bölümdeki U-Net'i `masks == 3` ile eğitin ve sol atriyumun sistol sonundaki Dice katsayısını bildirin. Yapısal kalp ekibi atriyumun büyüklüğüyle neden ilgilenir? |
| Kardiyovasküler görüntüleme | U-Net'i 200 yerine ilk 100 eğitim hastasıyla eğitin ve Dice katsayısını görüntüye ve faza göre karşılaştırın. Yeni bir laboratuvar kaç sınırı çizilmiş incelemeye ihtiyaç duyardı? |
| Ekokardiyografi laboratuvarı | `camus_table` tablosunu kullanarak iki görüntüsü de iyi kalitede olan hastaların EF hatasını diğer hastalarınkiyle karşılaştırın. Bir laboratuvar otomatik ölçümden önce kötü pencereleri dışlamalı mıdır ve bunu yaparsa neyi kaybeder? |
| Kardiyo-onkoloji | Kanser tedavisi sırasında yapılan bir kontrol ekokardiyogramı, EF'nin öncekinden 10 puan düşük olduğunu göstermektedir. 9. bölümdeki farkların standart sapmasını kullanın: Her biri uzmandan bağımsız olarak saptığında, değişmemiş bir kalbin iki otomatik ölçümü arasındaki fark ne kadar büyük olabilir? |
| Elektrofizyoloji | Atriyal fibrilasyonda RR aralığı atımdan atıma değişir; kılavuzlar bu yüzden ölçümün birkaç atımın ortalaması alınarak yapılmasını önerir. CAMUS her görüntü için bir diyastol sonu ve bir sistol sonu karesi verir. Hattı atriyal fibrilasyonlu hastalarda kullanmadan önce neyi değiştirirdiniz? |
| Yoğun bakım | `camus.to_original` ve `camus.measure` dâhil olmak üzere, U-Net'in bir hastanın dört karesi için ne kadar süreye ihtiyaç duyduğunu kendi bilgisayarınızda ölçün. Yoğun bakımda otomatik bir EF, inceleme biter bitmez yatak başında hazır olabilir mi? |

## Defterden sonra

[Etkileşimli labı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-04/lab.html) açın ve *Kendini sına* sekmesindeki öz değerlendirmeyi yapın. Ardından *Düşün ve dışa aktar* sekmesinde yansıtma yazınızı yazın ve öğrenme günlüğünü dışa aktarın. [Hafta sayfasında](README.md) anlatılan 4. haftanın haftalık projesinde CAMUS'un 50 test hastasının EF'si 7. bölümdeki U-Net ile ölçülür ve bu ölçüm bir ekokardiyografi laboratuvarının yapacağı gibi değerlendirilir.

<div align="center">

---

**Prof. Dr. Utku Köse**<br>
Profesör, Bilgisayar Mühendisliği Bölümü, Süleyman Demirel Üniversitesi, Isparta, Türkiye<br>
Kurucu Müdür, Yapay Zekâ Uygulama ve Araştırma Merkezi (YAZEM), Süleyman Demirel Üniversitesi<br>
Bilgisayar Bilimleri Anabilim Dalı Başkanı, Bilgisayar Mühendisliği Bölümü, Süleyman Demirel Üniversitesi<br>
Diğer kurumlar: University of North Dakota (ABD), Universidad Panamericana (Meksiko, Meksika), Vel Tech University (Chennai, Hindistan)<br>
IEEE Kıdemli Üyesi, ACM Profesyonel Üyesi

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

*Python ile Yapay Zekâ Eğitimi: Kardiyologlar için Uygulamalı Eğitim Programı, Türk Kardiyoloji Derneği. İlk sürüm: Kasım 2026. İçerik CC BY 4.0, kod MIT.*

</div>